<a href="https://colab.research.google.com/github/jaysulk/Poisson-GENERIC-Neural-Operators/blob/main/ICLR2027_gfno_testbed_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Poisson-GENERIC vs. GENERIC-FNO on GENERIC-FNO's own testbed
Companion to the ICLR 2027 rebuttal notebook: the same definitions (Sections 1–6), then one benchmark section. Run all cells; `QUICK_T` (one seed) takes ~20 min on a T4, three seeds ~1 h.


## 1. Setup and configuration

In [ ]:
import math, time, itertools, copy, os
from dataclasses import dataclass, asdict, field
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cuda.matmul.allow_tf32 = False   # keep the structural residuals honest
torch.backends.cudnn.allow_tf32 = False
print('device:', DEV, '| torch', torch.__version__)

def seed_all(s):
    torch.manual_seed(s); np.random.seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def sdims(dim):            # spatial dims of a (B, C, *grid) tensor
    return tuple(range(-dim, 0))

def ip(a, b):              # grid-measure inner product: sum over channels and space / number of grid points -> (B,)
    # NOTE: dividing by the number of grid points only (not channels) keeps the measure independent of the channel
    # count, so Q(u) on the p observed channels and E(z) on the p+1 channels of z use the SAME inner product.  With a
    # per-channel mean, the Riesz-scaled gradient of Q came out (p+1)/p x too large (3x for the wave), the exact
    # exponential integrated only 1/(p+1) of the linear flow and the learned symbols settled at 1/2 (scalar) and 1/3 (wave).
    return (a * b).flatten(1).sum(1) / a[0, 0].numel()

def ip_pointwise(a, b):    # inner product over the channel axis only -> (B, *grid)
    return (a * b).sum(1)

def nrm(a):
    return ip(a, a).sqrt()

def neg_index(a, dim):     # a[..., k] -> a[..., (-k) mod n] on the last `dim` axes (fftn layout)
    d = sdims(dim)
    return torch.roll(torch.flip(a, d), shifts=(1,) * dim, dims=d)

device: cuda | torch 2.14.1+cu130


In [ ]:
@dataclass
class Cfg:
    # problem
    dim: int = 1
    pde: str = 'heat'            # heat | advection | burgers | wave | landau | twostream (1D-1V Vlasov-Poisson on the (x,v) grid, dim=2)
    nx: int = 64
    n_traj: int = 120
    T: int = 20                  # saved frames per trajectory (beyond t=0)
    dt: float = 0.1              # frame spacing
    kmax: int = 8                # band limit of the initial conditions
    # model
    model: str = 'pg'            # 'pg' (Poisson-GENERIC) | 'base' (unconstrained residual operator) | 'cfo' (Continuous Flow Operator baseline)
                                 #   | 'gfno' (GENERIC-FNO, this paper's protocol) | 'gfno-native' (GENERIC-FNO with its own training recipe)
    backbone: str = 'fno'        # fno | transolver | cno
    width: int = 32              # backbone width (baseline) / functional width (pg)
    modes: int = 12              # FNO modes
    layers: int = 3
    norm: bool = False           # InstanceNorm inside FNO/CNO blocks (paper: needed in 2D, harmful in 1D)
    m_L: int = 16                # band of the Poisson multiplier  (|k|_inf <= m_L, k != 0)
    m_M: int = 16                # band of the dissipative multiplier
    K_mode: str = 'complement'   # kernel band the entropy sees: 'complement' (k=0 and |k|>m_L) | 'mean' (k=0 only)
    s_init: str = 'encoder'      # latent entropy density init: 'encoder' | 'zero'
    op_init: float = 0.1         # init scale of the multiplier symbols (too small -> model starts in the identity basin)
    energy: str = 'learned'      # 'learned': E = h(mean g(u,s)) | 'physical': E = Q(u) + mean e(s), Q fixed, e >= 0 learned
    entropy: str = 'learned'     # 'learned': Casimir functional of (Pi_K u, s) | 'integral': S = mean(s) (exact Casimir)
    m_type: str = 'constant'     # 'constant': D_M multiplier | 'state': A(z)A(z)^T with A = D_M o Gamma(z), Gamma a pointwise matrix field
    n_sub: int = 1               # split integrator: sub-steps for the explicit dissipative remainder (stability: coef*k_max^2/n_sub < 2)
    flux_init: bool = True       # state-M: initialize D_M as gradient/divergence flux slots (model starts as a nonlinear diffusion)
    flux_scale: float = 0.05     # scale of that init; explicit Euler on the dissipative part is stable only if coef*k_max^2 < 2
    lie_poisson: bool = False    # scalar fields: add lambda*(u D + D u), D = sum_a d_a, to L (Lie-Poisson, Poisson; compatible with D):
                                 #   nonlinear transport (Burgers = -(1/3)(uD+Du) delta/du 1/2||u||^2) with a QUADRATIC energy
    phi_gauge: str = 'proj'    # 'taylor': subtract phi's degree-2 Taylor polynomial at 0 (removes the quadratic gauge) | 'none': phi >= 0
    conserve_mass: str = 'auto'  # 'auto': for the scalar conservation laws (heat/advection/burgers) M gets no local (zeroth-order) slot on u,
                                 #   so int u is conserved and there is no Rayleigh-type damping of a conserved density | 'yes' | 'no'
    legacy_mask: bool = False    # reproduce the SUBMITTED conserve_mass mask (local-slot column of u instead of the u row); Sec. R1 only
    dg_gamma: str = 'midpoint'   # discrete-gradient step: evaluate Gamma(z) at the 'midpoint' (as submitted) or freeze it at the
                                 #   explicit 'predictor'.  Theorem 3 holds either way (P-bar is built from the discrete gradient);
                                 #   'predictor' takes the backbone out of the fixed-point map (Reviewer VbeA Q3).
    m_coords: bool = False       # Gamma(z, x): give the friction coefficient net the grid coordinates as well as z.  Needed in phase
                                 #   space: the Dougherty drift (v - u(x)) f is a function of the coordinate v, not of f alone
    sub: int = 1                 # sub-cycling of the euler / rk4 / dg frame step (z -> z + f/sub, `sub` times).  The phase-space
                                 #   bracket is a first-order transport operator: one explicit step per frame is CFL-unstable, the
                                 #   Vlasov runs use integrator='rk4', sub=4.  Distinct from n_sub (which belongs to 'split').
    # CFO baseline (Hou, Huang & Perdikaris, ICLR 2026): flow matching on quintic-Hermite trajectory splines, RK4 at inference
    cfo_gamma0: float = 1e-5     # noise amplitude gamma_0 of the stochastic interpolant (paper: 1e-5 / 1e-6; stable up to 1e-4)
    cfo_m: int = 3               # noise-schedule exponent (C^2 schedule, matches the quintic spline)
    cfo_K: int = 4               # random (segment, tau) samples per trajectory per optimizer step (= AR horizon: same number of targets)
    cfo_substeps: int = 1        # RK4 steps per frame at inference (4 NFE each; the autoregressive models use 1 NFE per frame)
    cfo_time_cond: bool = True   # N_theta(t, u): sinusoidal embedding of the normalized time t/(T dt) appended as input channels
    # training
    epochs: int = 40
    batch: int = 16
    horizon: int = 4             # training rollout horizon
    lr: float = 1e-3
    lr_op: float = 3e-2          # learning rate of the operator symbols a(k), B(k): the true advection symbol is a(k) = -c k dt
                                 #   (= -0.8 at k=8), unreachable from a 0.1 init in ~100 Adam steps at lr 1e-3
    wd: float = 1e-4
    # evaluation
    n_roll: int = 10
    integrator: str = 'euler'    # evaluation integrator: 'euler' | 'dg' (discrete gradient, exact E/S) | 'split' (exact unitary step
                                 #   for the linear reversible part exp(L Hess Q) + Euler for the rest; physical variant only)
    train_integrator: str = 'euler'   # training integrator ('dg' is exact but ~dg_iters x slower)
    dg_iters: int = 6
    seed: int = 0

PDE_PARAMS = {
    'heat':      dict(nu=0.05),
    'advection': dict(c=1.0),
    'burgers':   dict(nu=0.02),
    'wave':      dict(c=1.0, gamma=0.3),
    # 1D-1V Vlasov-Poisson, electrons on a neutralizing ion background:  f_t + v f_x - E f_v = C[f],  E_x = 1 - rho.
    #   k0: fundamental wavenumber (box L_x = 2 pi / k0); vmax: v in [-vmax, vmax), tapered to a periodic coordinate for |v| > vc
    #   (see VlasovGrid); nu: collision frequency; alpha: range of the
    #   density-perturbation amplitude; v0: range of the beam speed (0 = Maxwellian).  collision: 'dougherty' (conservative
    #   Lenard-Bernstein, nu d_v[(v-u)f + T d_v f]: conserves mass, momentum, energy; H-theorem) | 'lb' (thermostat, NOT conservative)
    'landau':    dict(k0=0.5, vmax=4 * math.pi, vc=2 * math.pi, nu=0.04, alpha=(0.05, 0.5), v0=(0.0, 0.0), collision='dougherty'),
    'twostream': dict(k0=0.2, vmax=5 * math.pi, vc=2.5 * math.pi, nu=0.005, alpha=(0.02, 0.1), v0=(2.0, 2.5), collision='dougherty'),
    #   twostream: Dougherty relaxes towards a Maxwellian at the TOTAL temperature 1 + v0^2 ~ 7 (energy is conserved), so its
    #   diffusion coefficient nu T is large; nu = 0.005 keeps the beams (diffusion length ~0.9 over t = 16) and the instability,
    #   and the wider box (vmax = 5 pi, vc = 2.5 pi) keeps the broadened beams inside |v| < vc.  Landau: Maxwellian tails are
    #   1e-9 at 2 pi, so the narrower box (finer dv, Nyquist k_v = 8) resolves the k0 t filamentation; nu = 0.04 damps it to 0.3%
    #   by t = 12 (amplitude exp(-nu k0^2 t^3 / 3)) -- at nu = 0.02 it is still 5% at k_v = 6 and the discrete S_B is not monotone.
}
VLASOV = ('landau', 'twostream')
def is_vlasov(pde): return pde in VLASOV
def n_fields(pde):           # observed channels p
    return 2 if pde == 'wave' else 1

## 2. Data
Spectral RK4 solver on the periodic box $[0,2\pi)^d$, band-limited random initial conditions (unit RMS). Frame spacing `dt = 0.1` is chosen so that persistence is *far* from the truth over the evaluation horizon — this addresses the near-persistence caveat of the first paper's ten-step tables. Damped wave is fully observed as `(q, v)` (the GENERIC-FNO partial-observation failure is deliberately not reproduced; set the observed channels yourself if you want it).

In [ ]:
class Spectral:
    """Spectral derivatives on the periodic box [0, 2pi)^d with integer wavenumbers."""
    def __init__(self, nx, dim, device=DEV, dtype=torch.float64):
        k = torch.fft.fftfreq(nx, d=1.0 / nx, device=device, dtype=dtype)
        self.ks = torch.meshgrid(*([k] * dim), indexing='ij')
        self.k2 = sum(kk ** 2 for kk in self.ks)
        kinf = torch.stack([kk.abs() for kk in self.ks]).amax(0)
        self.dealias_mask = (kinf < nx / 3.0)          # 2/3 rule
        self.dim = dim; self.d = sdims(dim)
    def fft(self, u):  return torch.fft.fftn(u, dim=self.d)
    def ifft(self, uh): return torch.fft.ifftn(uh, dim=self.d).real
    def deriv(self, u, axis):
        return self.ifft(1j * self.ks[axis] * self.fft(u))
    def lap(self, u):
        return self.ifft(-self.k2 * self.fft(u))
    def dealias(self, u):
        return self.ifft(self.dealias_mask * self.fft(u))

def pde_rhs(pde, u, sp, prm):
    """u: (B, p, *grid) float64. Returns du/dt."""
    if pde == 'heat':
        return prm['nu'] * sp.lap(u)
    if pde == 'advection':
        return -prm['c'] * sum(sp.deriv(u, a) for a in range(sp.dim))
    if pde == 'burgers':
        adv = sum(sp.deriv(u, a) for a in range(sp.dim))
        return -sp.dealias(u * adv) + prm['nu'] * sp.lap(u)
    if pde == 'wave':
        q, v = u[:, :1], u[:, 1:]
        return torch.cat([v, prm['c'] ** 2 * sp.lap(q) - prm['gamma'] * v], 1)
    raise ValueError(pde)

def rk4(f, u, dt):
    k1 = f(u); k2 = f(u + 0.5 * dt * k1); k3 = f(u + 0.5 * dt * k2); k4 = f(u + dt * k3)
    return u + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)

def random_field(n, nx, dim, kmax, device=DEV, dtype=torch.float64):
    """Band-limited random field with a mild red spectrum, unit RMS."""
    sp = Spectral(nx, dim, device, dtype)
    w = torch.randn(n, 1, *([nx] * dim), device=device, dtype=dtype)
    kinf = torch.stack([kk.abs() for kk in sp.ks]).amax(0)
    mask = (kinf <= kmax) & (kinf > 0)
    amp = 1.0 / (1.0 + sp.k2) ** 0.75
    u = sp.ifft(sp.fft(w) * mask * amp)
    return u / u.flatten(1).std(1).view(n, *([1] * (dim + 1)))

def n_substeps(cfg, nx):
    """RK4 sub-steps per frame from the linear stability limit at this resolution (nu k^2 dt, c k dt < ~2.8)."""
    prm = PDE_PARAMS[cfg.pde]; kmax = nx / 2
    if cfg.pde in ('heat', 'burgers'):
        dt_max = min(0.02, 2.0 / (prm['nu'] * cfg.dim * kmax ** 2))
    else:
        dt_max = min(0.02, 2.0 / (prm['c'] * cfg.dim * kmax))
    return max(1, int(math.ceil(cfg.dt / dt_max)))

def gen_data(cfg, device=DEV):
    """Returns (n_traj, T+1, p, *grid) float32 trajectories."""
    if is_vlasov(cfg.pde): return gen_vlasov(cfg, device=device)
    seed_all(cfg.seed)
    p = n_fields(cfg.pde); prm = PDE_PARAMS[cfg.pde]
    sp = Spectral(cfg.nx, cfg.dim, device)
    u = random_field(cfg.n_traj, cfg.nx, cfg.dim, cfg.kmax, device)
    if cfg.pde == 'wave':
        v = 0.5 * random_field(cfg.n_traj, cfg.nx, cfg.dim, cfg.kmax, device)
        u = torch.cat([u, v], 1)
    n_sub = n_substeps(cfg, cfg.nx); dt_sub = cfg.dt / n_sub
    f = lambda x: pde_rhs(cfg.pde, x, sp, prm)
    frames = [u]
    with torch.no_grad():
        for _ in range(cfg.T):
            for _ in range(n_sub):
                u = rk4(f, u, dt_sub)
            frames.append(u)
    return torch.stack(frames, 1).float()

def split_data(data, frac=0.8):
    n = int(frac * data.shape[0])
    return data[:n], data[n:]

### 2b. Vlasov–Poisson solver (1D-1V, spectral RK4, Dougherty collisions)

In [ ]:
def smoothstep(a, n=6):
    """C^n monotone step 0 -> 1 on [0, 1] (regularized incomplete beta I_a(n+1, n+1); polynomial of degree 2n+1)."""
    out = torch.zeros_like(a)
    for k in range(n + 1):
        out = out + math.comb(n + k, k) * math.comb(2 * n + 1, n - k) * (-a) ** k
    return a ** (n + 1) * out

def periodic_velocity(v, vmax, vc):
    """v~(v): equal to v for |v| <= vc, tapered smoothly (C^6) to 0 at |v| = vmax, hence an odd, periodic, smooth function on the
    v-grid.  The kinetic energy v~^2/2 is then a smooth periodic function whose spectral derivative is v to ~1e-5 for |v| < vc
    (v^2/2 itself has a derivative jump at the seam and its spectral derivative rings everywhere).  f is < 1e-3 of its peak
    where v~ != v (Landau, vc = 2 pi: Maxwellian 3e-9; two-stream, vc = 2.5 pi: beams at v0 <= 2.5 broadened to width 1.35 by
    collisions, 4e-4)."""
    a = ((v.abs() - vc) / (vmax - vc)).clamp(0.0, 1.0)
    return v * (1.0 - smoothstep(a))

class VlasovGrid:
    """1D-1V phase space (x, v) on the periodic box [0, L_x) x [-vmax, vmax), L_x = 2 pi / k0, nx x nx points, Fourier in both
    directions with PHYSICAL wavenumbers.  Axis -2 is x, axis -1 is v.  Electrons on a neutralizing ion background:
        f_t + v f_x - E f_v = C[f],    E_x = 1 - rho,  <E> = 0,   U_x = E   (U is the potential energy of an electron)
    Hamiltonian H[f] = int v^2/2 f + 1/2 int E^2,  dH/df = v^2/2 + U(x);  Lie-Poisson bracket {F,G} = int f {dF/df, dG/df}_{x,v}.
    Casimirs int Phi(f) for every Phi (mass, int f^2, -int f ln f, ...).  C[f] = Dougherty operator nu d_v[(v-u)f + T d_v f] with
    the local moments u(x), T(x): conserves mass, momentum and energy and has an H-theorem for S_B = -int f ln f -> metriplectic.
    The velocity coordinate enters everywhere (streaming, moments, H) as the periodic v~ of `periodic_velocity`, so the system
    solved here IS a smooth metriplectic system on the periodic grid with Hamiltonian H~ = int v~^2/2 f + 1/2 int E^2 -- the
    one the model's phase-space bracket represents -- and it differs from Vlasov-Poisson only where f < 1e-3 of its peak.
    De-aliasing (2/3 rule) in x only: the nonlinear products E f_v, (v~ - u) f, T f_v have x-dependent coefficients; nothing
    is truncated in v (the smooth v~^2/2 must keep its full spectrum)."""
    def __init__(self, nx, prm, device=DEV, dtype=torch.float64):
        self.nx, self.prm = nx, prm
        self.Lx, self.Lv = 2 * math.pi / prm['k0'], 2 * prm['vmax']
        self.dx, self.dv = self.Lx / nx, self.Lv / nx
        kx = 2 * math.pi * torch.fft.fftfreq(nx, d=self.dx, device=device, dtype=dtype)
        kv = 2 * math.pi * torch.fft.fftfreq(nx, d=self.dv, device=device, dtype=dtype)
        self.kx, self.kv = kx.view(nx, 1), kv.view(1, nx)
        self.x = (torch.arange(nx, device=device, dtype=dtype) * self.dx).view(1, 1, nx, 1)
        self.v = (torch.arange(nx, device=device, dtype=dtype) * self.dv - prm['vmax']).view(1, 1, 1, nx)   # for plots / extents
        self.vt = periodic_velocity(self.v, prm['vmax'], prm['vc'])                                            # used by the physics
        self.kappa = 0.5 * prm['vmax'] ** 2                                                                     # energy unit (see hamiltonian)
        cdt = torch.complex64 if dtype == torch.float32 else torch.complex128
        inv = torch.zeros(nx, 1, device=device, dtype=cdt)
        inv[1:] = 1.0 / (1j * self.kx[1:])
        self.inv_ik = inv.view(1, 1, nx, 1)                                   # (i k)^-1, zero at k = 0
        m = torch.fft.fftfreq(nx, d=1.0 / nx, device=device, dtype=dtype).abs()
        self.dealias = (m.view(nx, 1) < nx / 3.0)                           # 2/3 rule in x only
    def dx_(self, f): return torch.fft.ifft(1j * self.kx * torch.fft.fft(f, dim=-2), dim=-2).real
    def dv_(self, f): return torch.fft.ifft(1j * self.kv * torch.fft.fft(f, dim=-1), dim=-1).real
    def P(self, f):   return torch.fft.ifft2(torch.fft.fft2(f) * self.dealias).real
    def moments(self, f):                                            # rho, bulk velocity u, temperature T: (B,1,nx,1)
        rho = f.sum(-1, keepdim=True) * self.dv
        u = (f * self.vt).sum(-1, keepdim=True) * self.dv / rho.clamp_min(1e-12)
        T = ((self.vt - u) ** 2 * f).sum(-1, keepdim=True) * self.dv / rho.clamp_min(1e-12)
        return rho, u, T
    def field(self, f):                                              # E and U = d_x^{-1} E, both (B,1,nx,1)
        rho = f.sum(-1, keepdim=True) * self.dv
        rh = torch.fft.fft(1.0 - rho, dim=-2)
        Eh = rh * self.inv_ik; Uh = Eh * self.inv_ik
        return torch.fft.ifft(Eh, dim=-2).real, torch.fft.ifft(Uh, dim=-2).real
    def collision(self, f):
        nu = self.prm['nu']
        if self.prm.get('collision', 'dougherty') == 'lb':
            u, T = 0.0, 1.0                                          # plain Lenard-Bernstein: relaxes to a fixed Maxwellian, not conservative
        else:
            _, u, T = self.moments(f)
        return nu * self.dv_(self.P((self.vt - u) * f + T * self.dv_(f)))
    def rhs(self, f):
        E, _ = self.field(f)
        return -self.vt * self.dx_(f) + self.P(E * self.dv_(f)) + self.collision(f)
    def hamiltonian(self, f):
        """Fixed 'physical energy' in the notebook's grid-mean units:  Q = [ip(v~^2/2, f) + mean_x(E^2) / (2 L_v)] / kappa,  so that
        the Riesz gradient w.r.t. ip is exactly (v~^2/2 + U(x)) / kappa  (H_phys = kappa L_x L_v Q).  Plays the role of 1/2||u||^2
        of the scalar PDEs.  kappa = vmax^2/2 makes the gradient O(1) like the scalar PDEs' (v^2/2 reaches ~80 on this box, and
        the learned multipliers act on the gradient); the model's lambda absorbs it: lambda_exact = dt kappa (2 pi)^2 / (L_x L_v) ~ 1."""
        E, U = self.field(f)
        Q = (ip(0.5 * self.vt ** 2 * torch.ones_like(f), f) + (E ** 2).flatten(1).mean(1) / (2 * self.Lv)) / self.kappa
        return Q, ((0.5 * self.vt ** 2 + U) / self.kappa).expand_as(f)
    def invariants(self, f):                                         # physical units, each (B,)
        rho, u, T = self.moments(f); E, _ = self.field(f)
        vol = self.Lx * self.Lv
        fpos = f.clamp_min(1e-30)
        return dict(mass=vol * f.flatten(1).mean(1),
                    momentum=vol * (self.vt * f).flatten(1).mean(1),
                    H=vol * self.kappa * self.hamiltonian(f)[0],
                    W_E=0.5 * self.Lx * (E ** 2).flatten(1).mean(1),               # electric energy: the Landau-damping observable
                    enstrophy=vol * (f ** 2).flatten(1).mean(1),                   # a Casimir of the collisionless flow
                    S_B=-vol * (fpos * fpos.log()).flatten(1).mean(1),             # Boltzmann entropy: Casimir, produced by C[f]
                    f_min=f.flatten(1).amin(1))

_VGRID = {}
def vgrid(pde, nx, device, dtype):
    key = (pde, nx, str(device), dtype)
    if key not in _VGRID: _VGRID[key] = VlasovGrid(nx, PDE_PARAMS[pde], device, dtype)
    return _VGRID[key]

def vlasov_ic(cfg, device=DEV, dtype=torch.float64, alpha=None, v0=None):
    """f0 = F(v) (1 + delta(x)):  F a Maxwellian (v0 = 0) or two counter-streaming Maxwellians at +-v0; delta a random real
    field on the first `kmax` harmonics of k0 (red spectrum, random phases), rescaled to peak amplitude alpha.
    Normalized so that <rho>_x = 1 (neutrality: the periodic Poisson problem needs it)."""
    prm = PDE_PARAMS[cfg.pde]; g = VlasovGrid(cfg.nx, prm, device, dtype); n = cfg.n_traj
    M = lambda v: torch.exp(-0.5 * v ** 2) / math.sqrt(2 * math.pi)
    v0 = torch.empty(n, 1, 1, 1, device=device, dtype=dtype).uniform_(*prm['v0']) if v0 is None else torch.full((n, 1, 1, 1), float(v0), device=device, dtype=dtype)
    Fv = 0.5 * (M(g.v - v0) + M(g.v + v0))
    m = torch.fft.fftfreq(cfg.nx, d=1.0 / cfg.nx, device=device, dtype=dtype)
    mask = ((m.abs() >= 1) & (m.abs() <= cfg.kmax)).view(1, 1, -1) / (1.0 + m.abs()).view(1, 1, -1)
    w = torch.randn(n, 1, cfg.nx, device=device, dtype=dtype)
    d = torch.fft.ifft(torch.fft.fft(w, dim=-1) * mask, dim=-1).real
    a = torch.empty(n, 1, 1, device=device, dtype=dtype).uniform_(*prm['alpha']) if alpha is None else torch.full((n, 1, 1), float(alpha), device=device, dtype=dtype)
    d = a * d / d.abs().amax(-1, keepdim=True)
    f = Fv * (1.0 + d.unsqueeze(-1))
    rho_mean = (f.sum(-1) * g.dv).mean(-1, keepdim=True).unsqueeze(-1)   # (n,1,1,1)
    return f / rho_mean

def vlasov_substeps(cfg, nx):
    """RK4 sub-steps per frame from the explicit stability limits of the three terms (streaming v k_x, acceleration E k_v,
    collisional diffusion nu T k_v^2) at the Nyquist wavenumbers, with margin inside RK4's stability radius ~2.8."""
    prm = PDE_PARAMS[cfg.pde]; g = VlasovGrid(nx, prm, 'cpu')
    kxm, kvm = math.pi / g.dx, math.pi / g.dv                       # Nyquist: the streaming term v~ f_x is not truncated
    Tmax = 1.0 + prm['v0'][1] ** 2
    lam = float(g.vt.abs().max()) * kxm + 1.0 * kvm + prm['nu'] * Tmax * kvm ** 2
    return max(1, int(math.ceil(cfg.dt * lam / 2.0)))

def gen_vlasov(cfg, f0=None, device=DEV):
    """(n_traj, T+1, 1, nx, nx) float32 Vlasov-Poisson trajectories (RK4, float64, de-aliased spectral in x and v)."""
    seed_all(cfg.seed); prm = PDE_PARAMS[cfg.pde]
    f = vlasov_ic(cfg, device) if f0 is None else f0.double().to(device)
    g = VlasovGrid(f.shape[-1], prm, device)
    n_sub = vlasov_substeps(cfg, f.shape[-1]); dt_sub = cfg.dt / n_sub
    frames = [f]
    with torch.no_grad():
        for _ in range(cfg.T):
            for _ in range(n_sub): f = rk4(g.rhs, f, dt_sub)
            frames.append(f)
    return torch.stack(frames, 1).float()

LANDAU_THEORY = dict(k=0.5, omega=1.4156, gamma=0.1533)   # linear Landau damping of the k = 0.5 mode of a unit Maxwellian


## 3. Backbones
Three dimension-generic backbones with one interface `(B, C_in, *grid) → (B, C_out, *grid)`:
* **FNO** (Li et al., 2021) with optional InstanceNorm per block (the normalization the first paper diagnosed as necessary in 2D / harmful in 1D — `cfg.norm`);
* **Transolver** physics-attention (Wu et al., ICML 2024): grid points → learned slices → attention among slices → back. Dimension-agnostic on flattened tokens;
* **CNO-lite**: periodic convolutional U-shape in the spirit of CNO (Raonic et al., 2023), without the alias-free activation machinery.

All three serve both as the unconstrained baseline `u_{t+1} = u_t + G(u_t)` and as the backbone of the learned functionals `E_θ, S_φ` and the entropy encoder.

In [ ]:
def ConvNd(dim):  return {1: nn.Conv1d, 2: nn.Conv2d}[dim]
def INormNd(dim): return {1: nn.InstanceNorm1d, 2: nn.InstanceNorm2d}[dim]

def coord_grid(x):
    """(B, C, *grid) -> (B, dim, *grid) coordinates in [0,1)."""
    B, dim = x.shape[0], x.dim() - 2
    axes = [torch.linspace(0, 1, x.shape[2 + i] + 1, device=x.device, dtype=x.dtype)[:-1] for i in range(dim)]
    g = torch.stack(torch.meshgrid(*axes, indexing='ij'), 0)
    return g.unsqueeze(0).expand(B, *g.shape)

class SpectralConv(nn.Module):
    def __init__(self, dim, cin, cout, modes):
        super().__init__()
        self.dim, self.m = dim, modes
        scale = 1.0 / (cin * cout)
        shape = (cin, cout) + (modes,) * dim + (2,)
        self.w1 = nn.Parameter(scale * torch.randn(*shape))
        if dim == 2: self.w2 = nn.Parameter(scale * torch.randn(*shape))
    def forward(self, x):
        m = self.m
        if self.dim == 1:
            xf = torch.fft.rfft(x)
            out = torch.zeros(x.shape[0], self.w1.shape[1], xf.shape[-1], dtype=xf.dtype, device=x.device)
            out[:, :, :m] = torch.einsum('bim,iom->bom', xf[:, :, :m], torch.view_as_complex(self.w1))
            return torch.fft.irfft(out, n=x.shape[-1])
        xf = torch.fft.rfft2(x)
        out = torch.zeros(x.shape[0], self.w1.shape[1], *xf.shape[-2:], dtype=xf.dtype, device=x.device)
        out[:, :, :m, :m] = torch.einsum('bixy,ioxy->boxy', xf[:, :, :m, :m], torch.view_as_complex(self.w1))
        out[:, :, -m:, :m] = torch.einsum('bixy,ioxy->boxy', xf[:, :, -m:, :m], torch.view_as_complex(self.w2))
        return torch.fft.irfft2(out, s=x.shape[-2:])

class FNO(nn.Module):
    """Fourier Neural Operator (Li et al., 2021), dimension-generic, optional InstanceNorm per block."""
    def __init__(self, dim, cin, cout, width=32, modes=12, layers=3, norm=False):
        super().__init__()
        C = ConvNd(dim)
        self.lift = C(cin + dim, width, 1)
        self.specs = nn.ModuleList([SpectralConv(dim, width, width, modes) for _ in range(layers)])
        self.ws = nn.ModuleList([C(width, width, 1) for _ in range(layers)])
        self.norms = nn.ModuleList([INormNd(dim)(width, affine=True) if norm else nn.Identity() for _ in range(layers)])
        self.proj = nn.Sequential(C(width, 2 * width, 1), nn.GELU(), C(2 * width, cout, 1))
    def forward(self, x):
        x = self.lift(torch.cat([x, coord_grid(x)], 1))
        for s, w, n in zip(self.specs, self.ws, self.norms):
            x = F.gelu(n(s(x) + w(x)))
        return self.proj(x)

class PhysicsAttention(nn.Module):
    """Transolver physics-attention (Wu et al., ICML 2024): tokens -> M learned slices -> attention -> back."""
    def __init__(self, C, heads=4, slices=32):
        super().__init__()
        self.h, self.s, self.ch = heads, slices, C // heads
        self.in_proj = nn.Linear(C, C)
        self.slice_proj = nn.Linear(C, heads * slices)
        self.temp = nn.Parameter(torch.tensor(0.5))
        self.qkv = nn.Linear(self.ch, 3 * self.ch)
        self.out = nn.Linear(C, C)
    def forward(self, x):                       # x: (B, N, C)
        B, N, C = x.shape
        fx = self.in_proj(x).view(B, N, self.h, self.ch)
        w = torch.softmax(self.slice_proj(x).view(B, N, self.h, self.s) / self.temp.clamp(min=0.05), dim=-1)
        tok = torch.einsum('bnhs,bnhc->bhsc', w, fx) / (w.sum(1).unsqueeze(-1) + 1e-6)   # (B,h,S,ch)
        q, k, v = self.qkv(tok).chunk(3, -1)
        att = torch.softmax(q @ k.transpose(-1, -2) / math.sqrt(self.ch), -1)
        tok = att @ v
        y = torch.einsum('bnhs,bhsc->bnhc', w, tok).reshape(B, N, C)
        return self.out(y)

class Transolver(nn.Module):
    def __init__(self, dim, cin, cout, width=64, layers=3, heads=4, slices=32):
        super().__init__()
        self.dim = dim
        self.lift = nn.Sequential(nn.Linear(cin + dim, width), nn.GELU(), nn.Linear(width, width))
        self.blocks = nn.ModuleList()
        for _ in range(layers):
            self.blocks.append(nn.ModuleDict(dict(
                ln1=nn.LayerNorm(width), att=PhysicsAttention(width, heads, slices),
                ln2=nn.LayerNorm(width),
                mlp=nn.Sequential(nn.Linear(width, 2 * width), nn.GELU(), nn.Linear(2 * width, width)))))
        self.proj = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, cout))
    def forward(self, x):
        B, C, *grid = x.shape
        t = torch.cat([x, coord_grid(x)], 1).flatten(2).transpose(1, 2)      # (B, N, C+dim)
        t = self.lift(t)
        for b in self.blocks:
            t = t + b['att'](b['ln1'](t))
            t = t + b['mlp'](b['ln2'](t))
        y = self.proj(t).transpose(1, 2).reshape(B, -1, *grid)
        return y

class CNO(nn.Module):
    """CNO-style periodic convolutional operator (Raonic et al., 2023), simplified: 2-level U-shape,
    circular padding, GELU; no alias-free up/down-sampled activations (hence '-lite')."""
    def __init__(self, dim, cin, cout, width=32, layers=2, norm=False):
        super().__init__()
        C = ConvNd(dim); self.dim = dim
        def block(ci, co):
            mods = [C(ci, co, 3, padding=1, padding_mode='circular')]
            if norm: mods.append(INormNd(dim)(co, affine=True))
            mods.append(nn.GELU())
            return nn.Sequential(*mods)
        self.lift = C(cin + dim, width, 1)
        self.enc1 = nn.Sequential(*[block(width, width) for _ in range(layers)])
        self.enc2 = nn.Sequential(block(width, 2 * width), *[block(2 * width, 2 * width) for _ in range(layers - 1)])
        self.dec = nn.Sequential(block(3 * width, width), *[block(width, width) for _ in range(layers - 1)])
        self.proj = C(width, cout, 1)
        self.pool = {1: F.avg_pool1d, 2: F.avg_pool2d}[dim]
        self.mode = {1: 'linear', 2: 'bilinear'}[dim]
    def forward(self, x):
        x = self.lift(torch.cat([x, coord_grid(x)], 1))
        h1 = self.enc1(x)
        h2 = self.enc2(self.pool(h1, 2))
        up = F.interpolate(h2, size=h1.shape[2:], mode=self.mode, align_corners=False)
        return self.proj(self.dec(torch.cat([h1, up], 1)))

def make_backbone(cfg, cin, cout, width=None):
    w = width or cfg.width
    if cfg.backbone == 'fno':
        return FNO(cfg.dim, cin, cout, w, cfg.modes, cfg.layers, cfg.norm)
    if cfg.backbone == 'transolver':
        return Transolver(cfg.dim, cin, cout, max(w, 32), cfg.layers)
    if cfg.backbone == 'cno':
        return CNO(cfg.dim, cin, cout, w, max(cfg.layers - 1, 1), cfg.norm)
    raise ValueError(cfg.backbone)

def zero_last_layer(net):
    last = [m for m in net.modules() if isinstance(m, (nn.Linear, nn.Conv1d, nn.Conv2d))][-1]
    nn.init.zeros_(last.weight); nn.init.zeros_(last.bias)

def count_params(m): return sum(p.numel() for p in m.parameters())

### 3b. CFO: a continuous-time baseline on the same backbones

In [ ]:
def time_embedding(t, n_bands=16):
    """Sinusoidal embedding of a normalized time t in [0,1] -> (B, 2 n_bands)."""
    freqs = (2.0 ** torch.arange(n_bands, device=t.device, dtype=t.dtype)) * math.pi
    ang = t.view(-1, 1) * freqs
    return torch.cat([ang.sin(), ang.cos()], -1)

class CFO(nn.Module):
    """Continuous Flow Operator (Hou, Huang & Perdikaris, ICLR 2026) on the notebook's backbones.
    Training: through the frames of every trajectory a C^2 quintic-Hermite spline s(t) (knot values, first derivatives by the
    2nd-order central stencil, second derivatives by the 3-point stencil; one-sided at the ends), the stochastic interpolant
    I(t) = s(t) + gamma(t) z,  z ~ N(0, I),  gamma = gamma_0 tau^m (1 - tau)^m on each segment (zero at the knots), and the
    flow-matching loss  E_{t, z} || N_theta(t, I(t)) - (s'(t) + gamma'(t) z) ||^2  with t ~ U[0, T dt].  The network thus learns
    the continuous-time right-hand side directly, without backpropagating through an ODE solver.
    Inference: RK4 on du/dt = N_theta(t, u), `cfo_substeps` RK4 steps per frame.  Time enters as a sinusoidal embedding of the
    normalized time, appended as constant channels (the backbones are shared with the other models and take (B, C, *grid)).
    Differences from the paper: uniform frame spacing only (their irregular-time-grid experiments are not reproduced), the
    notebook's backbones/optimizer/schedule instead of their U-Nets and 60k Adam steps, and the loss is normalized by the
    target energy of the batch so that it is O(1) like the relative-L2 losses of the other models."""
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg; self.p = n_fields(cfg.pde); self.nb = 16 if cfg.cfo_time_cond else 0
        self.net = make_backbone(cfg, self.p + 2 * self.nb, self.p)
        self.t_scale = cfg.T * cfg.dt
    def rhs(self, t, u):                                             # t: (B,) physical time
        if self.nb:
            e = time_embedding(t / self.t_scale, self.nb).to(u.dtype)
            e = e.view(*e.shape, *([1] * (u.dim() - 2))).expand(-1, -1, *u.shape[2:])
            u = torch.cat([u, e], 1)
        return self.net(u)
    # ---- quintic Hermite spline through uniformly spaced frames ----
    @staticmethod
    def knot_derivs(traj, dt):
        d = torch.zeros_like(traj); a = torch.zeros_like(traj)
        d[:, 1:-1] = (traj[:, 2:] - traj[:, :-2]) / (2 * dt)
        d[:, 0] = (-3 * traj[:, 0] + 4 * traj[:, 1] - traj[:, 2]) / (2 * dt)
        d[:, -1] = (3 * traj[:, -1] - 4 * traj[:, -2] + traj[:, -3]) / (2 * dt)
        a[:, 1:-1] = (traj[:, 2:] - 2 * traj[:, 1:-1] + traj[:, :-2]) / dt ** 2
        a[:, 0] = a[:, 1]; a[:, -1] = a[:, -2]
        return d, a
    @staticmethod
    def hermite(tau):
        """Quintic Hermite basis and its tau-derivative: value, h*d, h^2*a at the left knot, then the same at the right knot."""
        t2, t3, t4, t5 = tau ** 2, tau ** 3, tau ** 4, tau ** 5
        H = [1 - 10 * t3 + 15 * t4 - 6 * t5, tau - 6 * t3 + 8 * t4 - 3 * t5, 0.5 * t2 - 1.5 * t3 + 1.5 * t4 - 0.5 * t5,
             10 * t3 - 15 * t4 + 6 * t5, -4 * t3 + 7 * t4 - 3 * t5, 0.5 * t3 - t4 + 0.5 * t5]
        dH = [-30 * t2 + 60 * t3 - 30 * t4, 1 - 18 * t2 + 32 * t3 - 15 * t4, tau - 4.5 * t2 + 6 * t3 - 2.5 * t4,
              30 * t2 - 60 * t3 + 30 * t4, -12 * t2 + 28 * t3 - 15 * t4, 1.5 * t2 - 4 * t3 + 2.5 * t4]
        return H, dH
    def spline(self, traj, idx, tau):
        """s(t), s'(t) at t = (idx + tau) dt for the trajectories traj (B, N, p, *grid); idx, tau: (B,)."""
        dt = self.cfg.dt; d, a = self.knot_derivs(traj, dt)
        bi = torch.arange(traj.shape[0], device=traj.device)
        c = [traj[bi, idx], dt * d[bi, idx], dt ** 2 * a[bi, idx], traj[bi, idx + 1], dt * d[bi, idx + 1], dt ** 2 * a[bi, idx + 1]]
        H, dH = self.hermite(tau.to(traj.dtype))
        sh = (-1,) + (1,) * (traj.dim() - 2)
        s = sum(h.view(sh) * ci for h, ci in zip(H, c))
        ds = sum(h.view(sh) * ci for h, ci in zip(dH, c)) / dt
        return s, ds
    def fm_loss(self, traj):
        cfg = self.cfg; B, N = traj.shape[:2]; K = cfg.cfo_K; m = cfg.cfo_m
        trajK = traj.repeat_interleave(K, 0)                           # K independent (segment, tau) draws per trajectory
        idx = torch.randint(0, N - 1, (B * K,), device=traj.device)
        tau = torch.rand(B * K, device=traj.device, dtype=traj.dtype)
        s, ds = self.spline(trajK, idx, tau)
        sh = (-1,) + (1,) * (traj.dim() - 2)
        gamma = cfg.cfo_gamma0 * tau ** m * (1 - tau) ** m
        dgamma = cfg.cfo_gamma0 * m * (tau ** (m - 1) * (1 - tau) ** m - tau ** m * (1 - tau) ** (m - 1)) / cfg.dt
        z = torch.randn_like(s)
        I = s + gamma.view(sh) * z; target = ds + dgamma.view(sh) * z
        pred = self.rhs((idx.to(tau.dtype) + tau) * cfg.dt, I)
        return F.mse_loss(pred, target) / (target.pow(2).mean() + 1e-12)
    def rollout(self, u0, n, create_graph=True, keep_aux=False, integrator=None):
        cfg = self.cfg; u = u0; us = []; B = u0.shape[0]; h = cfg.dt / cfg.cfo_substeps
        for k in range(n):
            for j in range(cfg.cfo_substeps):
                t0 = torch.full((B,), (k * cfg.cfo_substeps + j) * h, device=u.device, dtype=u.dtype)
                k1 = self.rhs(t0, u); k2 = self.rhs(t0 + h / 2, u + h / 2 * k1)
                k3 = self.rhs(t0 + h / 2, u + h / 2 * k2); k4 = self.rhs(t0 + h, u + h * k3)
                u = u + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
            us.append(u)
        return torch.stack(us, 1), None, []


## 4. Operators
**Poisson operator.** Scalar field: symbol $i\,a(k)$ with $a$ made odd explicitly, $a(0)=0$, zero outside the band $|k|_\infty\le m_L$. Wave `(q,v)`: canonical $J=\begin{pmatrix}0&I\\-I&0\end{pmatrix}$ composed with an even non-negative multiplier. Both constant ⇒ Jacobi holds trivially (Prop. 1). `project_K` is $\Pi_K$, the projector onto the kernel band the entropy is allowed to see.

**Dissipative operator.** Per-mode Hermitian PSD matrix symbol $B(k)B(k)^H$ over all channels of $z$ (so friction can couple `v → s`), symmetrized for reality, sandwiched by the exact rank-one projection off $\delta E/\delta z$.

In [ ]:
def _scatter_band(param, nx, dim):
    """param: (2m+1,)*dim tensor indexed by k in [-m..m]  ->  (nx,)*dim tensor in fftn layout."""
    m = (param.shape[-1] - 1) // 2
    assert m < nx // 2, f'band m={m} must be < nx/2={nx//2}'
    idx = torch.arange(-m, m + 1, device=param.device) % nx
    full = torch.zeros(*([nx] * dim), device=param.device, dtype=param.dtype)
    if dim == 1:
        full[idx] = param
    else:
        full[idx[:, None], idx[None, :]] = param
    return full

class PoissonOperator(nn.Module):
    """Constant-coefficient skew Fourier multiplier on the mechanical block -> Poisson (Jacobi trivially).
    Scalar field:  symbol i a(k), a odd, a=0 on the kernel band K.
    Wave (q,v):    canonical J composed with an even, non-negative multiplier d(k) (zero on K)."""
    def __init__(self, dim, p, m, K_mode='complement', canonical=False, init_scale=0.05, vlasov=False):
        super().__init__()
        self.dim, self.p, self.m, self.K_mode, self.canonical, self.vlasov = dim, p, m, K_mode, canonical, vlasov
        self.a = nn.Parameter(init_scale * torch.randn(*([2 * m + 1] * dim)))
        self.lam = nn.Parameter(torch.zeros(1))            # Lie-Poisson coefficient (scalar fields only; 0 = off)
        self.use_lp = False
        # KdV pencil: when the Lie-Poisson term is on, the constant part is restricted to alpha1 D + alpha3 D^3 (D = sum_a d_a),
        # which is COMPATIBLE with u D + D u (the bi-Hamiltonian pair of KdV), so the full L is Poisson for all parameters.
        # A generic multiplier a(k) is not compatible with the Lie-Poisson term (Jacobi fails at O(0.1)), see Sec. 7.
        self.alpha = nn.Parameter(torch.tensor([init_scale, 0.0]) if not vlasov else 0.01 * init_scale * torch.randn(2))   # Vlasov: exact value 0
        self._cache = {}
    def symbol(self, nx):
        if self.use_lp:                                    # pencil: i(alpha1 kD - alpha3 kD^3) on the band, kD = sum_a k_a
            k = torch.fft.fftfreq(nx, d=1.0 / nx, device=self.a.device)
            ks = torch.meshgrid(*([k] * self.dim), indexing='ij'); kD = sum(ks)
            kinf = torch.stack([kk.abs() for kk in ks]).amax(0); band = (kinf <= self.m) & (kinf > 0)
            # alpha3 D^3 is compatible too but its k^3 growth wrecks the discrete-gradient fixed point and the explicit
            # remainder at high k; none of the test PDEs is dispersive, so the pencil is restricted to alpha1 D + lambda(uD+Du).
            if self.vlasov:                                # phase space: {a x + b v, .} = a d_v - b d_x are the constant multipliers
                kD = self.alpha[0] * ks[1] - self.alpha[1] * ks[0]   #   compatible with the Lie-Poisson bracket (L_{f + f0}, f0 linear)
                return 1j * kD * band
            return 1j * (self.alpha[0] * kD) * band
        full = _scatter_band(self.a, nx, self.dim)
        if self.canonical:
            d2 = full ** 2
            sym = 0.5 * (d2 + neg_index(d2, self.dim))                 # even, >= 0
            sym = sym.clone(); sym[(0,) * self.dim] = 0.0              # kernel contains k=0
            return sym.to(torch.complex64 if full.dtype == torch.float32 else torch.complex128)
        odd = 0.5 * (full - neg_index(full, self.dim))                 # odd  -> real skew operator
        return 1j * odd
    def K_mask(self, nx, device):
        key = (nx, str(device))
        if key not in self._cache:
            k = torch.fft.fftfreq(nx, d=1.0 / nx, device=device)
            ks = torch.meshgrid(*([k] * self.dim), indexing='ij')
            kinf = torch.stack([kk.abs() for kk in ks]).amax(0)
            zero = (kinf == 0)
            mask = zero if self.K_mode == 'mean' else (zero | (kinf > self.m))
            self._cache[key] = mask
        return self._cache[key]
    def project_K(self, u):                      # Pi_K: orthogonal projection onto the kernel band
        d = sdims(self.dim)
        return torch.fft.ifftn(torch.fft.fftn(u, dim=d) * self.K_mask(u.shape[-1], u.device), dim=d).real
    def exp_linear(self, u, cfg, dt=1.0):
        """Exact flow of  u_t = L dQ/du  for the FIXED quadratic energy Q over time dt: a unitary per-mode map,
        exp(dt * L Hess Q).  Scalar: multiply u_hat(k) by exp(i dt a(k)) (|.|=1 -> Q conserved exactly).
        Wave (q,v), Q = 1/2(v^2 + c^2 |grad q|^2):  per-mode 2x2 exponential of [[0, d],[-d c^2 |k|^2, 0]]."""
        d = sdims(self.dim); uf = torch.fft.fftn(u, dim=d); sym = self.symbol(u.shape[-1])
        if not self.canonical:
            return torch.fft.ifftn(uf * torch.exp(dt * sym), dim=d).real
        nx = u.shape[-1]; k = torch.fft.fftfreq(nx, d=1.0 / nx, device=u.device, dtype=u.dtype)
        k2 = sum(kk ** 2 for kk in torch.meshgrid(*([k] * self.dim), indexing='ij'))
        dsym = sym.real; c2 = PDE_PARAMS['wave']['c'] ** 2
        w = dsym * (c2 * k2).sqrt()                                       # oscillation frequency per mode
        cw, sw = torch.cos(dt * w), torch.sin(dt * w)
        ratio = torch.where(w > 0, dsym / w.clamp_min(1e-30), torch.zeros_like(w))   # d/omega (q <- v coupling)
        inv = torch.where(w > 0, w / dsym.clamp_min(1e-30), torch.zeros_like(w))    # omega/d (v <- q coupling)
        q, v = uf[:, 0], uf[:, 1]
        q1 = cw * q + ratio * sw * v
        v1 = -inv * sw * q + cw * v
        return torch.fft.ifftn(torch.stack([q1, v1], 1), dim=d).real
    def phase_space_bracket(self, f, g):
        """Vlasov Lie-Poisson operator  L_f g = -{f, g}_{x,v} = -(f_x g_v - f_v g_x)  (axis -2 = x, axis -1 = v; integer grid
        wavenumbers, the box lengths are absorbed in lambda).  Discretized as the average of the two continuum-equivalent forms
           A(g) = f_x g_v - f_v g_x,     D(g) = d_v(g f_x) - d_x(g f_v),
        because  <h, D g> = -<g, A h>  holds EXACTLY on the grid (spectral derivatives are skew matrices, pointwise products
        commute), so (A + D)/2 is exactly skew-adjoint for every f -- the phase-space analogue of the uD + Du form below.
        Jacobi holds in the continuum (Lie-Poisson) and on the grid in the resolved regime (band-limited states, 3b <= nx/2),
        with the same aliasing caveat as the scalar term; verified numerically in Sec. 12.  dH/df = v~^2/2 + U gives exactly
        -v~ f_x + E f_v (v~ the periodic velocity of VlasovGrid).  Any int Phi(f) is a Casimir: -{f, Phi'(f)} = 0 pointwise."""
        d = sdims(self.dim); nx = f.shape[-1]
        k = torch.fft.fftfreq(nx, d=1.0 / nx, device=f.device, dtype=f.dtype)
        mask = (k.abs().view(nx, 1) < nx / 3.0)                   # 2/3 rule in x only (see VlasovGrid: v keeps its full spectrum)
        Dx = lambda y: torch.fft.ifft(1j * k.view(nx, 1) * torch.fft.fft(y, dim=-2), dim=-2).real
        Dv = lambda y: torch.fft.ifft(1j * k.view(1, nx) * torch.fft.fft(y, dim=-1), dim=-1).real
        P = lambda y: torch.fft.ifftn(torch.fft.fftn(y, dim=d) * mask, dim=d).real
        f = P(f); g = P(g); fx, fv = Dx(f), Dv(f)
        A = fx * Dv(g) - fv * Dx(g)
        Dd = Dv(g * fx) - Dx(g * fv)
        return -0.5 * self.lam * P(A + Dd)
    def lie_poisson(self, u, g):
        """lambda (u D g + D(u g)),  D = sum_a d_a: skew-adjoint on the grid (pointwise products are symmetric, D is skew)
        and a Hamiltonian operator (Lie-Poisson bracket of vector fields along D); the diagnostics of Sec. 7 verify Jacobi
        for the full L = constant multiplier + Lie-Poisson term numerically."""
        if self.vlasov: return self.phase_space_bracket(u, g)
        d = sdims(self.dim); nx = u.shape[-1]
        k = torch.fft.fftfreq(nx, d=1.0 / nx, device=u.device, dtype=u.dtype)
        ks = torch.meshgrid(*([k] * self.dim), indexing='ij'); Dsym = 1j * sum(ks)
        kinf = torch.stack([kk.abs() for kk in ks]).amax(0); mask = (kinf < nx / 3.0)          # 2/3-rule projector P
        D = lambda x: torch.fft.ifftn(torch.fft.fftn(x, dim=d) * Dsym, dim=d).real
        P = lambda x: torch.fft.ifftn(torch.fft.fftn(x, dim=d) * mask, dim=d).real
        # P (u D + D u) P : the sandwich keeps skewness (P symmetric) and removes quadratic aliasing.  The discrete
        # operator is Poisson only in the RESOLVED regime: Jacobi involves triple products, and holds to machine
        # precision when the fields' band b satisfies 3b <= nx/2 (see lie_poisson_jacobi_vs_resolution), but fails at
        # O(0.1) on generic full-spectrum states.  This is why the Lie-Poisson term is OFF by default: with it, exact
        # Jacobi for any state is lost, and Burgers' nonlinear transport is the price (Sec. 6 of the paper).
        u = P(u); g = P(g)
        return self.lam * P(u * D(g) + D(u * g))
    def forward(self, g, u=None):                # g: (B, p, *grid) gradient wrt u  ->  L_u g   (u needed for the LP term)
        d = sdims(self.dim); sym = self.symbol(g.shape[-1])
        mul = lambda x: torch.fft.ifftn(torch.fft.fftn(x, dim=d) * sym, dim=d).real
        if not self.canonical:
            out = mul(g)
            if self.use_lp and u is not None: out = out + self.lie_poisson(u, g)
            return out
        gq, gv = g[:, :1], g[:, 1:]
        return torch.cat([mul(gv), -mul(gq)], 1)  # J D

class DissipativeOperator(nn.Module):
    """M = (I - P_E) D_M (I - P_E), D_M a per-mode Hermitian PSD matrix symbol B(k)B(k)^H on all channels of z."""
    def __init__(self, dim, c, m, init_scale=0.05):
        super().__init__()
        self.dim, self.c, self.m = dim, c, m
        self.B = nn.Parameter(init_scale * torch.randn(c, c, *([2 * m + 1] * dim), 2))
    def symbol(self, nx):
        Bc = torch.view_as_complex(self.B)
        full = torch.stack([torch.stack([_scatter_band(Bc[i, j], nx, self.dim) for j in range(self.c)]) for i in range(self.c)])
        S = torch.einsum('ij...,kj...->ik...', full, full.conj())            # B B^H  (Hermitian PSD per mode)
        return 0.5 * (S + neg_index(S, self.dim).conj())                      # reality: S(-k) = conj S(k)
    def raw(self, v):
        d = sdims(self.dim); S = self.symbol(v.shape[-1])
        vf = torch.fft.fftn(v, dim=d)
        return torch.fft.ifftn(torch.einsum('ij...,bj...->bi...', S, vf), dim=d).real
    @staticmethod
    def proj_off(w, v, tiny=1e-30):              # (I - P_w) v   (exact ratio; guarded only against w == 0)
        coef = ip(w, v) / ip(w, w).clamp_min(tiny)
        return v - coef.view(-1, *([1] * (v.dim() - 1))) * w
    def forward(self, v, w):                     # M v, with degeneracy direction w = dE/dz
        return self.proj_off(w, self.raw(self.proj_off(w, v)))

## 5. Model
`z = (u, s)`; `s` is a latent entropy density (encoder from `u`, or zero). `S_φ` receives `Π_K u` instead of `u` ⇒ `δS/δu ∈ span(K) = ker D_L`, so `L δS/δz = 0` for any parameters. Two integrators: explicit Euler (cheap, `O(Δt²)` drift) and the discrete-gradient step of §4.6 / Theorem 2 (exact discrete first and second laws).

In [ ]:
class Functional(nn.Module):
    """Scalar functional  F[z] = h( mean_x g(z) ),  g a neural operator backbone. Optional input projector."""
    def __init__(self, cfg, cin, proj=None):
        super().__init__()
        self.net = make_backbone(cfg, cin, cfg.width)
        self.head = nn.Sequential(nn.Linear(cfg.width, 64), nn.GELU(), nn.Linear(64, 1))
        self.proj = proj
    def forward(self, z, p):
        u, s = z[:, :p], z[:, p:]
        if self.proj is not None:
            u = self.proj(u)
        f = self.net(torch.cat([u, s], 1))
        return self.head(f.mean(sdims(z.dim() - 2))).squeeze(-1)

class PhysicalEnergy(nn.Module):
    """E[z] = Q(u) + mean_x phi(u) + mean_x e(s):
       Q      fixed quadratic mechanical energy (paper-1 diagnostic energy; for wave 1/2(v^2 + c^2|grad q|^2)),
       phi>=0 learned pointwise potential on u  (needed for nonlinear reversible transport: Burgers = d_x delta/du int u^3/6),
       e >=0  learned convex, increasing internal energy of the entropy density, so T = e'(s) > 0.
       E is conserved exactly and phi, e >= 0  =>  Q(u_t) <= E[z_0] for all t (bounded physical energy by construction)."""
    def __init__(self, cfg, p):
        super().__init__(); self.cfg = cfg; self.p = p
        self.phi = nn.Sequential(nn.Linear(p, 32), nn.GELU(), nn.Linear(32, 32), nn.GELU(), nn.Linear(32, 1))
        self.w_raw = nn.Parameter(-3.0 * torch.ones(16)); self.a_raw = nn.Parameter(torch.randn(16) * 0.5); self.b = nn.Parameter(torch.randn(16))
    def _quad_basis(self, x):                                            # monomials of degree <= 2 in the p channels
        cols = [torch.ones_like(x[..., 0])] + [x[..., i] for i in range(self.p)]
        cols += [x[..., i] * x[..., j] for i in range(self.p) for j in range(i, self.p)]
        return torch.stack(cols, -1)
    def potential(self, u):                                              # (B, *grid)
        """phi(u) = psi(u) - Pi_quad psi(u):  psi = softplus(MLP) >= 0, minus its weighted least-squares projection onto
        polynomials of degree <= 2, computed on a fixed Gaussian quadrature grid over the data range.  This removes the
        quadratic gauge GLOBALLY (a Taylor subtraction at u=0 does not: the network can hide a quadratic away from the
        origin, and did).  Without it phi absorbs alpha*Q, L's symbol settles at (1-alpha) x exact and the explicit
        remainder step carries the rest.  phi is not sign-definite; the bound reads Q(u_t) <= E(z_0) - |Omega| inf phi."""
        x = u.movedim(1, -1)
        psi = F.softplus(self.phi(x)).squeeze(-1)
        if getattr(self.cfg, 'phi_gauge', 'proj') == 'none':
            return psi
        if not hasattr(self, '_nodes') or self._nodes.device != u.device or self._nodes.dtype != u.dtype:
            g = torch.linspace(-4, 4, 17, device=u.device, dtype=u.dtype)
            nodes = torch.stack(torch.meshgrid(*([g] * self.p), indexing='ij'), -1).reshape(-1, self.p)
            wts = torch.exp(-0.5 * (nodes ** 2).sum(-1))                           # unit-variance Gaussian reference measure
            self._nodes, self._wts = nodes, wts
        Phi = self._quad_basis(self._nodes)                                         # (n, nb)
        psi_n = F.softplus(self.phi(self._nodes)).squeeze(-1)                       # (n,)
        W = self._wts
        A = Phi.T @ (Phi * W.unsqueeze(-1)); bvec = Phi.T @ (psi_n * W)                # weighted normal equations
        coef = torch.linalg.solve(A + 1e-8 * torch.eye(A.shape[0], device=A.device, dtype=A.dtype), bvec)
        return psi - (self._quad_basis(x) * coef).sum(-1)
    T0 = 0.1   # temperature floor: e(s) = T0 s + e_tilde(s), e_tilde convex >= 0, so T = e'(s) >= T0.  Physical friction
               # coefficients carry 1/T; without a floor, regions where s drifts negative send T -> 0 and the explicit
               # dissipative step blows up.  The energy bound survives via the second law (S non-decreasing):
               #   Q(u_t) <= E(z_0) - T0 S(z_0) - |Omega| inf phi.
    def internal(self, s):                                               # (B, *grid), convex increasing in s
        w, a = F.softplus(self.w_raw), F.softplus(self.a_raw)
        return self.T0 * s.squeeze(1) + (w * F.softplus(a * s.unsqueeze(-1) + self.b)).sum(-1).squeeze(1)
    def temperature(self, s):                                            # T = e'(s) >= T0
        w, a = F.softplus(self.w_raw), F.softplus(self.a_raw)
        return self.T0 + (w * a * torch.sigmoid(a * s.unsqueeze(-1) + self.b)).sum(-1)
    def forward(self, z, p):
        Q, _ = phys_energy(z[:, :p], self.cfg)
        return Q + self.potential(z[:, :p]).flatten(1).mean(1) + self.internal(z[:, p:]).flatten(1).mean(1)

class IntegralEntropy(nn.Module):
    """S[z] = mean_x s : the textbook Casimir. dS/dz = (0, 1) exactly."""
    def forward(self, z, p):
        return z[:, p:].flatten(1).mean(1)

class StateDissipativeOperator(DissipativeOperator):
    """M = (I-P_E) A A^T (I-P_E),  A = D_M o P(x) o Gamma(z),  with an ONSAGER PARITY structure.
    Slots (r = c*(d+1)): for each channel i, d 'flux' slots (i,a) and one 'local' slot i.
      D_M   : channel i <- flux slot (j,a) via  -i k_a * b_{ij a}(k)  (a divergence times an even real symbol),
              channel i <- local slot j  via  b'_{ij}(k) (even real symbol).  Derivative parity is fixed by construction.
      Gamma : pointwise r x r.  flux-flux and local-local blocks are arbitrary pointwise functions of z;
              flux<->local blocks are LINEAR in the gradient features (grad z)(x) with pointwise coefficients.
    Hence every path channel <- channel through M has EVEN total derivative order: M can diffuse and damp but cannot
    transport.  Without this the dissipative channel can carry advection (M(0,1)_u = -d_x(c u)), and the learned
    reversible symbol splits the transport with it (observed: exactly 1/2 of the exact a(k), 1/3 of d(k) for wave).
    The closed-form heat / damped-wave friction operators are exactly of this form (verify_representability)."""
    def __init__(self, cfg, c, m, p, init_scale=0.05, conserve_mass=False):
        nn.Module.__init__(self)
        self.dim, self.c, self.m, self.p, self.conserve_mass = cfg.dim, c, m, p, conserve_mass
        d = cfg.dim; self.d = d; self.nf = c * d; self.r = c * d + c
        # even real symbols on the band: b (c, c, d) for flux slots, b' (c, c) for local slots
        self.b = nn.Parameter(init_scale * torch.randn(c, c, d, *([2 * m + 1] * d)))
        self.bl = nn.Parameter(init_scale * torch.randn(c, c, *([2 * m + 1] * d)))
        with torch.no_grad():
            if cfg.flux_init:
                for i in range(c):
                    for a in range(d): self.b[i, i, a] += cfg.flux_scale
                    self.bl[i, i] += cfg.flux_scale
        # channel masks: no local slot into conserved-density channels (mass: the u-row of D_M has flux columns only, so
        #   (M v)_u is a divergence for every v), no friction at all on a wave's q
        rowmask = torch.ones(c); locmask = torch.ones(c)
        if cfg.pde == 'wave': rowmask[0] = 0.0
        if conserve_mass: locmask[:p] = 0.0
        self.register_buffer('row_mask', rowmask); self.register_buffer('loc_mask', locmask)
        # pointwise coefficient network on z: flux-flux block (nf x nf), local-local block (c x c),
        # cross coefficients h (nf x c x nf) and h' (c x nf x nf) multiplying gradient features
        nff, ncc, nx1, nx2 = self.nf ** 2, c ** 2, self.nf * c * self.nf, c * self.nf * self.nf
        self.m_coords = bool(getattr(cfg, 'm_coords', False))
        self.legacy_mask = bool(getattr(cfg, 'legacy_mask', False))          # Sec. R1: measure the submitted code's mass leak
        self.coef = nn.Sequential(nn.Linear(c + (d if self.m_coords else 0), 64), nn.GELU(), nn.Linear(64, 64), nn.GELU(), nn.Linear(64, nff + ncc + nx1 + nx2))
        nn.init.zeros_(self.coef[-1].weight); nn.init.zeros_(self.coef[-1].bias)
        self.gamma = None; self._cache = {}
    # ---- symbols ----
    def _even(self, prm, nx):
        full = torch.stack([_scatter_band(prm[idx], nx, self.dim) for idx in itertools.product(*[range(n) for n in prm.shape[:-self.dim]])])
        full = full.view(*prm.shape[:-self.dim], *([nx] * self.dim))
        return 0.5 * (full + neg_index(full, self.dim))
    def symbol_B(self, nx):
        """c x r complex symbol of D_M in fftn layout (columns: flux slots (j,a) then local slots j)."""
        key = (nx, str(self.b.device))
        if key not in self._cache:
            k = torch.fft.fftfreq(nx, d=1.0 / nx, device=self.b.device)
            self._cache[key] = torch.meshgrid(*([k] * self.dim), indexing='ij')
        ks = self._cache[key]
        be = self._even(self.b, nx); ble = self._even(self.bl, nx)                    # (c,c,d,*g), (c,c,*g)
        # saturated symbols: |flux symbol| < 1 and |local symbol| < 1 for ANY learned b, b'. Since M = A A^T for any A this
        # keeps symmetry/PSD/degeneracy exactly, and it bounds the gain of the dissipative operator so the explicit
        # sub-stepped remainder cannot go unstable at high k when b(k) drifts (14/72 seeds diverged without it).
        # For the physical heat symbol sqrt(nu) k the saturation is negligible on the data band ((sqrt(nu) k)^2 ~ 0.3 at k=8).
        kb = torch.stack([ks[a] * be[:, :, a] for a in range(self.d)], 2)
        flux = -1j * kb / torch.sqrt(1.0 + kb ** 2)                                     # (c, c, d, *g)
        flux = flux.reshape(self.c, self.nf, *([nx] * self.dim))
        view = (1, self.c) if self.legacy_mask else (self.c, 1)                      # legacy: the u local-slot COLUMN (submitted code)
        loc = ((ble / torch.sqrt(1.0 + ble ** 2)) * self.loc_mask.view(*view, *([1] * self.dim))).to(flux.dtype)   # BUG FIX: mask the
        #   OUTPUT row of the conserved channels (no local slot INTO u), not the u local-slot column; see Sec. 7 'mass' row
        B = torch.cat([flux, loc], 1) * self.row_mask.view(self.c, 1, *([1] * self.dim))
        return B
    # ---- state ----
    def grad_features(self, z):                                                          # (B, c*d, *grid): d_a z_j
        d = sdims(self.dim); nx = z.shape[-1]; ks = self._cache[(nx, str(z.device))] if (nx, str(z.device)) in self._cache else None
        if ks is None:
            self.symbol_B(nx); ks = self._cache[(nx, str(z.device))]
        zf = torch.fft.fftn(z, dim=d)
        g = torch.stack([torch.fft.ifftn(1j * ks[a] * zf, dim=d).real for a in range(self.d)], 2)   # (B, c, d, *g)
        return g.reshape(z.shape[0], self.nf, *z.shape[2:])
    def set_state(self, z, gE, gamma=None):
        Bsym = self.symbol_B(z.shape[-1])
        w = self._mul(gE, Bsym.conj().transpose(0, 1))
        self.w_hat = w / (ip_pointwise(w, w).sqrt().unsqueeze(1) + 1e-30)
        if gamma is not None: self.gamma = gamma; return
        B, c, nf, r = z.shape[0], self.c, self.nf, self.r
        zin = torch.cat([z, coord_grid(z)], 1) if self.m_coords else z                      # Gamma(z) or Gamma(z, x)
        co = (torch.tanh(self.coef(zin.movedim(1, -1))) / self.r).movedim(-1, 1)         # bounded so |Gamma - I| <= 1:
        # unbounded pointwise coefficients let the s-channel's local dynamics stiffen (s grew 40%/step and blew up the
        # explicit remainder step on advection); physical friction coefficients are bounded functions of the state
        i0 = 0
        ff = co[:, i0:i0 + nf * nf].view(B, nf, nf, *z.shape[2:]); i0 += nf * nf
        ll = co[:, i0:i0 + c * c].view(B, c, c, *z.shape[2:]); i0 += c * c
        h = co[:, i0:i0 + nf * c * nf].view(B, nf, c, nf, *z.shape[2:]); i0 += nf * c * nf
        hp = co[:, i0:i0 + c * nf * nf].view(B, c, nf, nf, *z.shape[2:])
        gz = self.grad_features(z)                                                       # (B, nf, *grid)
        G = torch.zeros(B, r, r, *z.shape[2:], device=z.device, dtype=z.dtype)
        eye_f = torch.eye(nf, device=z.device, dtype=z.dtype).view(1, nf, nf, *([1] * self.dim))
        eye_l = torch.eye(c, device=z.device, dtype=z.dtype).view(1, c, c, *([1] * self.dim))
        G[:, :nf, :nf] = eye_f + ff
        G[:, nf:, nf:] = eye_l + ll
        G[:, :nf, nf:] = torch.einsum('bijk...,bk...->bij...', h, gz)                    # flux <- local: linear in grad z
        G[:, nf:, :nf] = torch.einsum('bijk...,bk...->bij...', hp, gz)                   # local <- flux: linear in grad z
        self.gamma = G
    def _mul(self, v, Bsym):
        d = sdims(self.dim); vf = torch.fft.fftn(v, dim=d)
        return torch.fft.ifftn(torch.einsum('ij...,bj...->bi...', Bsym, vf), dim=d).real
    def _P(self, w):
        return w - ip_pointwise(self.w_hat, w).unsqueeze(1) * self.w_hat
    def raw(self, v):                           # A A^T v = D_M P Gamma Gamma^T P D_M^H v
        Bsym = self.symbol_B(v.shape[-1])
        w = self._P(self._mul(v, Bsym.conj().transpose(0, 1)))
        w = torch.einsum('bji...,bj...->bi...', self.gamma, w)
        w = torch.einsum('bij...,bj...->bi...', self.gamma, w)
        return self._mul(self._P(w), Bsym)
    def forward(self, v, w, project=False):
        if not project: return self.raw(v)
        return self.proj_off(w, self.raw(self.proj_off(w, v)))

class PoissonGENERIC(nn.Module):
    """dz/dt = L dE/dz + M dS/dz  on z = (u, s):  L constant Poisson, S a Casimir of L by construction,
    M PSD with the projection sandwich.  Full metriplectic structure, exact for any parameters."""
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg; self.p = n_fields(cfg.pde); c = self.p + 1
        self.L = PoissonOperator(cfg.dim, self.p, cfg.m_L, cfg.K_mode, canonical=(cfg.pde == 'wave'), init_scale=cfg.op_init,
                                 vlasov=is_vlasov(cfg.pde))
        self.L.use_lp = bool(getattr(cfg, 'lie_poisson', False)) and cfg.pde != 'wave' and cfg.energy == 'physical'
        cm = (cfg.pde != 'wave') if cfg.conserve_mass == 'auto' else (cfg.conserve_mass == 'yes')   # scalar conservation laws conserve int u
        self.M = (StateDissipativeOperator(cfg, c, cfg.m_M, self.p, init_scale=cfg.op_init, conserve_mass=cm) if cfg.m_type == 'state'
                  else DissipativeOperator(cfg.dim, c, cfg.m_M, init_scale=cfg.op_init))
        self.E = PhysicalEnergy(cfg, self.p) if cfg.energy == 'physical' else Functional(cfg, c)
        self.S = IntegralEntropy() if cfg.entropy == 'integral' else Functional(cfg, c, proj=self.L.project_K)
        self.enc = make_backbone(cfg, self.p, 1, width=max(cfg.width // 2, 8)) if cfg.s_init == 'encoder' else None
    # ---- state ----
    def init_state(self, u):
        s = self.enc(u) if self.enc is not None else torch.zeros_like(u[:, :1])
        return torch.cat([u, s], 1)
    def Lz(self, g, z=None):                     # L acts on the u-block only; s-block of L is zero
        u = z[:, :self.p] if z is not None else None
        return torch.cat([self.L(g[:, :self.p], u), torch.zeros_like(g[:, self.p:])], 1)
    def Lz_const(self, g):                       # constant-multiplier part of L only (no Lie-Poisson term)
        return torch.cat([self.L(g[:, :self.p], None), torch.zeros_like(g[:, self.p:])], 1)
    def PiK_z(self, d):                          # projection onto ker L  (K modes of u, all of s)
        return torch.cat([self.L.project_K(d[:, :self.p]), d[:, self.p:]], 1)
    def grads(self, z, create_graph=True):
        with torch.enable_grad():
            if not z.requires_grad: z = z.requires_grad_(True)
            E = self.E(z, self.p); S = self.S(z, self.p)
            N = z[0, 0].numel()   # Riesz representative w.r.t. the grid-measure inner product: (#grid points) x the autodiff gradient
            gE = N * torch.autograd.grad(E.sum(), z, create_graph=create_graph)[0]
            gS = N * torch.autograd.grad(S.sum(), z, create_graph=create_graph)[0]
        return E, S, gE, gS
    def vector_field(self, z, create_graph=True):
        E, S, gE, gS = self.grads(z, create_graph)
        if isinstance(self.M, StateDissipativeOperator): self.M.set_state(z, gE)
        f = self.Lz(gE, z) + self.M(gS, gE)
        return f, dict(E=E, S=S, gE=gE, gS=gS)
    # ---- integrators ----
    def step_euler(self, z, create_graph=True, scale=1.0):
        f, aux = self.vector_field(z, create_graph)
        return z + scale * f, aux
    def step_rk4(self, z, create_graph=True, scale=1.0):
        """Classical RK4 on the learned metriplectic vector field (the frame map is exp(scale * f)).  Not exactly conservative
        (O(scale^5) per step) but stable for first-order transport operators such as the phase-space bracket at CFL < 2.8;
        the DG step gives the exact laws.  aux is from the frame's start state."""
        k1, aux = self.vector_field(z, create_graph); k2, _ = self.vector_field(z + 0.5 * scale * k1, create_graph)
        k3, _ = self.vector_field(z + 0.5 * scale * k2, create_graph); k4, _ = self.vector_field(z + scale * k3, create_graph)
        return z + scale / 6 * (k1 + 2 * k2 + 2 * k3 + k4), aux
    def step_dg(self, z, iters=None, create_graph=False, scale=1.0):
        """Discrete-gradient (Gonzalez) step. dE/dz uses the standard correction; dS/dz uses the correction
        restricted to ker L, so L gS_bar = 0 still holds and both laws are exact in discrete time (Thm 2)."""
        iters = iters or self.cfg.dg_iters
        z1, aux = self.step_euler(z, create_graph, scale)
        E0 = aux['E']; S0 = aux['S']
        gam = None
        if isinstance(self.M, StateDissipativeOperator) and getattr(self.cfg, 'dg_gamma', 'midpoint') == 'predictor':
            self.M.set_state(z1, aux['gE']); gam = self.M.gamma               # Gamma frozen at the predictor; P-bar still from gE-bar below
        for it in range(iters):
            zm = 0.5 * (z + z1); d = z1 - z
            E1, S1 = self.E(z1, self.p), self.S(z1, self.p)
            _, _, gE, gS = self.grads(zm, create_graph)
            if isinstance(self.M, StateDissipativeOperator): self.M.set_state(zm, gEb if it > 0 else gE, gamma=gam)
            cE = (E1 - E0 - ip(gE, d)) / (ip(d, d) + 1e-30)
            gEb = gE + cE.view(-1, *([1] * (z.dim() - 1))) * d
            dK = self.PiK_z(d)
            cS = (S1 - S0 - ip(gS, d)) / (ip(dK, dK) + 1e-30)
            gSb = gS + cS.view(-1, *([1] * (z.dim() - 1))) * dK
            z_new = z + scale * (self.Lz(gEb, zm) + self.M(gSb, gEb))
            z1 = z_new if it == iters - 1 else 0.5 * (z1 + z_new)     # relaxed iterations, plain final update
        aux.update(dict(gEbar=gEb, gSbar=gSb))
        return z1, aux
    def step_split(self, z, create_graph=True):
        """Strang split: exact unitary half-step of the linear reversible flow (conserves Q exactly), Euler step of the
        remainder (nonlinear potential + dissipation), exact half-step again.  Physical-energy variant only."""
        assert isinstance(self.E, PhysicalEnergy), 'split integrator needs the fixed quadratic energy'
        assert not is_vlasov(self.cfg.pde), 'split integrator needs Q = 1/2||u||^2 (exp_linear); use rk4 or dg with sub-cycling for Vlasov'
        p = self.p
        u = self.L.exp_linear(z[:, :p], self.cfg, 0.5); z = torch.cat([u, z[:, p:]], 1)
        f, aux = self.vector_field(z, create_graph)
        _, gQ = phys_energy(z[:, :p], self.cfg)                            # linear part of dE/du, already integrated exactly
        f = f - self.Lz_const(torch.cat([gQ, torch.zeros_like(z[:, p:])], 1))   # subtract ONLY the constant-multiplier part
        z = z + f / self.cfg.n_sub
        for _ in range(self.cfg.n_sub - 1):                                # optional sub-cycling of the stiff dissipative remainder
            f, _ = self.vector_field(z, create_graph); _, gQ = phys_energy(z[:, :p], self.cfg)
            z = z + (f - self.Lz_const(torch.cat([gQ, torch.zeros_like(z[:, p:])], 1))) / self.cfg.n_sub
        u = self.L.exp_linear(z[:, :p], self.cfg, 0.5); z = torch.cat([u, z[:, p:]], 1)
        return z, aux
    def step(self, z, create_graph=True, integrator=None):
        integrator = integrator or self.cfg.integrator
        if integrator == 'split':
            return self.step_split(z, create_graph)
        n = max(1, int(getattr(self.cfg, 'sub', 1))); aux0 = None
        for _ in range(n):                                   # sub-cycling (sub = 1 reproduces the original single step)
            if integrator == 'dg':    z, aux = self.step_dg(z, create_graph=create_graph, scale=1.0 / n)
            elif integrator == 'rk4': z, aux = self.step_rk4(z, create_graph, scale=1.0 / n)
            else:                     z, aux = self.step_euler(z, create_graph, scale=1.0 / n)
            if aux0 is None: aux0 = aux
        return z, aux0
    def rollout(self, u0, n, create_graph=True, keep_aux=False, integrator=None):
        z = self.init_state(u0); us, zs, auxs = [], [z], []
        for _ in range(n):
            z, aux = self.step(z, create_graph, integrator)
            if not create_graph: z = z.detach()
            us.append(z[:, :self.p]); zs.append(z)
            if keep_aux: auxs.append({k: v.detach() for k, v in aux.items()})
        return torch.stack(us, 1), torch.stack(zs, 1), auxs

class Baseline(nn.Module):
    """Unconstrained residual operator  u_{t+1} = u_t + G(u_t)."""
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg; self.p = n_fields(cfg.pde)
        self.net = make_backbone(cfg, self.p, self.p)
    def rollout(self, u0, n, create_graph=True, keep_aux=False, integrator=None):
        u = u0; us = []
        for _ in range(n):
            u = u + self.net(u); us.append(u)
        return torch.stack(us, 1), None, []

def make_model(cfg):
    return {'pg': PoissonGENERIC, 'base': Baseline, 'cfo': CFO, 'gfno': GENERICFNO, 'gfno-native': GENERICFNO}[cfg.model](cfg).to(DEV)

### 5b. GENERIC-FNO (Sulskis & Ravi, 2026), ported from its public code

In [ ]:
class GFNOBackbone(nn.Module):
    """GENERIC-FNO's own FNO backbone (github.com/jaysulk/GENERIC-FNO, `FNO_Backbone` / `FNO_Backbone2d`): 1x1 lift (no
    coordinate channels) -> [spectral conv + 1x1 skip -> InstanceNorm -> GELU] x layers -> 1x1 -> GELU -> 1x1.  Dimension-generic
    through this notebook's SpectralConv, which uses the same rfft / two-corner layout."""
    def __init__(self, dim, cin, cout, width=24, modes=12, layers=3):
        super().__init__()
        C = ConvNd(dim)
        self.lift = C(cin, width, 1)
        self.specs = nn.ModuleList([SpectralConv(dim, width, width, modes) for _ in range(layers)])
        self.skips = nn.ModuleList([C(width, width, 1) for _ in range(layers)])
        self.norms = nn.ModuleList([INormNd(dim)(width) for _ in range(layers)])        # affine=False, as in the original
        self.proj = nn.Sequential(C(width, width, 1), nn.GELU(), C(width, cout, 1))
    def forward(self, x):
        x = self.lift(x)
        for s, w, n in zip(self.specs, self.skips, self.norms):
            x = F.gelu(n(s(x) + w(x)))
        return self.proj(x)

class GFNOFunctional(nn.Module):
    """GENERIC-FNO's `FunctionalNet`: backbone -> density field -> spatial mean -> Linear(1,16)-GELU-Linear(16,1) -> scalar."""
    def __init__(self, dim, p, width=24, modes=12, layers=3):
        super().__init__()
        self.backbone = GFNOBackbone(dim, p, 1, width, modes, layers)
        self.head = nn.Sequential(nn.Linear(1, 16), nn.GELU(), nn.Linear(16, 1))
    def forward(self, u):
        return self.head(self.backbone(u).flatten(1).mean(1, keepdim=True)).squeeze(-1)      # (B,)

class GENERICFNO(nn.Module):
    """GENERIC-FNO (Sulskis & Ravi, TMLR 2026): the degeneracy-by-construction model of record, ported from the public code to
    this notebook's interface.  E and S are learned FNO functionals;  L = (I - P_S) D_L (I - P_S),  M = (I - P_E) D_M (I - P_E)
    with diagonal Fourier multipliers D_L = i a(k), D_M = |b(k)|^2 on the lowest `m_L` modes per direction (two k_x corners in
    2D) and P_w the rank-one projection onto w;  one explicit Euler step  u_{t+1} = u_t + L dE/du + M dS/du  (dt absorbed in
    the symbols; Euclidean autograd gradient, as in the original: the removals are scale-invariant and the symbols absorb the
    scale).  Exact by construction: L dS = 0, M dE = 0, hence <dE, du/dt> = 0 and <dS, du/dt> = <dS, M dS> >= 0.  Its L is
    state-dependent through P_S and does not satisfy Jacobi (Table 1 of the paper, 'projected L' row).
    For p > 1 channels (the (q,v) wave, which GENERIC-FNO did not treat) the multipliers are per channel and the removals act on
    the whole state; p = 1 reproduces the published model.  The original detaches u before each step; under this paper's
    protocol the rollout is differentiated through (see `train_gfno_native` for the original recipe)."""
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg; self.p = n_fields(cfg.pde); self.dim = cfg.dim; self.m = cfg.m_L
        mf = min(12, cfg.nx // 2)                                    # the original's modes_func = 12 (nx >= 24 in every cell)
        self.E_net = GFNOFunctional(cfg.dim, self.p, 24, mf, 3)
        self.S_net = GFNOFunctional(cfg.dim, self.p, 24, mf, 3)
        shape = (self.p, self.m) if cfg.dim == 1 else (self.p, self.m, self.m)
        self.a = nn.Parameter(0.3 * torch.randn(*shape)); self.b_r = nn.Parameter(0.3 * torch.randn(*shape)); self.b_i = nn.Parameter(0.3 * torch.randn(*shape))
        if cfg.dim == 2:                                            # second k_x corner (negative k_x), as in GENERIC_FNO2d
            self.a_n = nn.Parameter(0.3 * torch.randn(*shape)); self.b_rn = nn.Parameter(0.3 * torch.randn(*shape)); self.b_in = nn.Parameter(0.3 * torch.randn(*shape))
    def _mult(self, v, sym, sym_n=None):
        m = self.m
        if self.dim == 1:
            vh = torch.fft.rfft(v); out = torch.zeros_like(vh)
            out[:, :, :m] = sym.unsqueeze(0) * vh[:, :, :m]
            return torch.fft.irfft(out, n=v.shape[-1])
        vh = torch.fft.rfft2(v); out = torch.zeros_like(vh)
        out[:, :, :m, :m] = sym.unsqueeze(0) * vh[:, :, :m, :m]
        out[:, :, -m:, :m] = sym_n.unsqueeze(0) * vh[:, :, -m:, :m]
        return torch.fft.irfft2(out, s=v.shape[-2:])
    def L_apply(self, v): return self._mult(v, 1j * self.a, None if self.dim == 1 else 1j * self.a_n)
    def M_apply(self, v): return self._mult(v, self.b_r ** 2 + self.b_i ** 2, None if self.dim == 1 else self.b_rn ** 2 + self.b_in ** 2)
    @staticmethod
    def remove(v, w):                                               # (I - P_w) v, per sample, over all channels and grid points
        sh = (-1,) + (1,) * (v.dim() - 1)
        return v - ((v * w).flatten(1).sum(1) / ((w * w).flatten(1).sum(1) + 1e-12)).view(sh) * w
    def grads(self, u, create_graph=True):
        with torch.enable_grad():
            if not u.requires_grad: u = u.requires_grad_(True)
            E = self.E_net(u); S = self.S_net(u)
            gE = torch.autograd.grad(E.sum(), u, create_graph=create_graph)[0]
            gS = torch.autograd.grad(S.sum(), u, create_graph=create_graph)[0]
        return E, S, gE, gS
    def vector_field(self, u, create_graph=True):
        E, S, gE, gS = self.grads(u, create_graph)
        rev = self.remove(self.L_apply(self.remove(gE, gS)), gS)
        diss = self.remove(self.M_apply(self.remove(gS, gE)), gE)
        return rev + diss, dict(E=E, S=S, gE=gE, gS=gS)
    def step(self, u, create_graph=True):
        f, aux = self.vector_field(u, create_graph); return u + f, aux
    def rollout(self, u0, n, create_graph=True, keep_aux=False, integrator=None):
        u = u0; us = []; auxs = []
        for _ in range(n):
            u, aux = self.step(u, create_graph)
            if not create_graph: u = u.detach()
            us.append(u)
            if keep_aux: auxs.append({k: v.detach() for k, v in aux.items()})
        return torch.stack(us, 1), None, auxs
    def diagnostics(self, u0, pred):
        """GENERIC-FNO's own structural numbers along a rollout: first-order energy rate r_E = |<dE, du>| / (||dE|| ||du||)
        (what the construction makes zero), relative change of its learned E per step, minimum change of its learned S."""
        us = torch.cat([u0[:, None], pred], 1); rE, dE, dS = [], [], []
        for t in range(pred.shape[1]):
            E0, S0, gE, _ = self.grads(us[:, t], create_graph=False); E1, S1, _, _ = self.grads(us[:, t + 1], create_graph=False)
            du = us[:, t + 1] - us[:, t]
            rE.append((ip(gE, du).abs() / (nrm(gE) * nrm(du) + 1e-30)).mean().item())
            dE.append(((E1 - E0).abs() / (E0.abs() + 1e-12)).mean().item()); dS.append((S1 - S0).min().item())
        return dict(rE_own=float(np.mean(rE)), dE_own_rel=float(np.mean(dE)), min_dS_own=float(np.min(dS)))

def train_gfno_native(model, data, cfg, verbose=True):
    """GENERIC-FNO's own training recipe (its `train_model`, 1D defaults): AdamW(1e-3, wd 1e-4), cosine schedule, batch 32,
    one-step MSE on a random frame pair per batch; after epoch 30, with probability 0.3, a 3-step rollout MSE in which u is
    detached before every step (the original forward detaches its input); gradient clipping at 1; 150 epochs in 1D, 120 in 2D.
    No functional supervision (`e_sup_mode='none'`), no degeneracy penalty (exact by construction)."""
    n_epochs = 150 if cfg.dim == 1 else 120; bs = 32 if cfg.dim == 1 else 16
    if getattr(cfg, 'gfno_native_epochs', None): n_epochs = cfg.gfno_native_epochs
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, n_epochs)
    nt = data.shape[1] - 1; model.train(); t0 = time.time(); hist = []
    for ep in range(n_epochs):
        perm = torch.randperm(data.shape[0]); tot = 0.0; nb = 0
        for i in range(0, data.shape[0], bs):
            traj = data[perm[i:i + bs]].to(DEV)
            if ep > 30 and np.random.rand() < 0.3:
                t_start = np.random.randint(0, max(1, nt - 2)); L_ = min(3, nt - t_start); pred = traj[:, t_start]; loss = 0.0
                for s_ in range(L_):
                    pred, _ = model.step(pred.detach(), create_graph=True)
                    loss = loss + F.mse_loss(pred, traj[:, t_start + s_ + 1])
                loss = loss / L_
            else:
                t_idx = np.random.randint(0, nt)
                pred, _ = model.step(traj[:, t_idx].detach(), create_graph=True)
                loss = F.mse_loss(pred, traj[:, t_idx + 1])
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); tot += loss.item(); nb += 1
        sched.step(); hist.append(tot / nb)
        if verbose and (ep % max(1, n_epochs // 5) == 0 or ep == n_epochs - 1):
            print(f'  ep {ep:3d}  mse {tot/nb:.6f}  ({time.time()-t0:.0f}s)')
    return hist


## 6. Training and evaluation
Loss: relative $L^2$ on the observed field over a short rollout from a random start (same protocol for baseline and Poisson-GENERIC). Diagnostics: one-step / rollout / final error, persistence reference, gauge-invariant $r_{\rm mech}$ against the ground-truth rate $\Pi^\star$ (both computed from the fixed physical energy), and for PG the residuals $r_E$, $|ΔE|/|E|$, $\min ΔS$.

In [ ]:
def phys_energy(u, cfg):
    """Fixed quadratic physical energy Q(u) and its gradient dQ/du (no learned quantity involved)."""
    if is_vlasov(cfg.pde):                                   # Vlasov-Poisson Hamiltonian (linear + quadratic in f), gradient v^2/2 + U
        return vgrid(cfg.pde, u.shape[-1], u.device, u.dtype).hamiltonian(u)
    if cfg.pde != 'wave':
        return 0.5 * ip(u, u), u
    sp = Spectral(u.shape[-1], cfg.dim, u.device, u.dtype); c2 = PDE_PARAMS['wave']['c'] ** 2
    q, v = u[:, :1], u[:, 1:]
    Q = 0.5 * (ip(v, v) - c2 * ip(q, sp.lap(q)))
    return Q, torch.cat([-c2 * sp.lap(q), v], 1)

def rel_l2(pred, true):                          # per-sample relative L2, averaged -> scalar
    return (nrm(pred - true) / (nrm(true) + 1e-12)).mean()

def train_model(model, data, cfg, verbose=True):
    if cfg.model == 'gfno-native': return train_gfno_native(model, data, cfg, verbose)   # GENERIC-FNO's own recipe (Sec. R3)
    op_names = {'L.a', 'L.lam', 'L.alpha', 'M.b', 'M.bl'}   # operator symbols get the large lr; the friction symbols are saturated (|.|<1), so this is safe
    op_params = [p for n, p in model.named_parameters() if n in op_names]
    other = [p for n, p in model.named_parameters() if n not in op_names]
    groups = [dict(params=other, lr=cfg.lr)] + ([dict(params=op_params, lr=cfg.lr_op, weight_decay=0.0)] if op_params else [])
    opt = torch.optim.AdamW(groups, lr=cfg.lr, weight_decay=cfg.wd)
    n_iter = cfg.epochs * math.ceil(data.shape[0] / cfg.batch)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[g['lr'] for g in groups], total_steps=n_iter, pct_start=0.1)
    H = cfg.horizon; model.train(); t0 = time.time(); hist = []
    for ep in range(cfg.epochs):
        perm = torch.randperm(data.shape[0]); tot = 0.0; nb = 0
        for i in range(0, data.shape[0], cfg.batch):
            traj = data[perm[i:i + cfg.batch]].to(DEV)
            if isinstance(model, CFO):                       # flow matching on the whole trajectory (no rollout in training)
                loss = model.fm_loss(traj)
            else:
                t_start = np.random.randint(0, cfg.T - H + 1)
                u0 = traj[:, t_start]; target = traj[:, t_start + 1:t_start + 1 + H]
                pred, _, _ = model.rollout(u0, H, create_graph=True, integrator=cfg.train_integrator)
                loss = sum(rel_l2(pred[:, h], target[:, h]) for h in range(H)) / H
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); tot += loss.item(); nb += 1
        hist.append(tot / nb)
        if verbose and (ep % max(1, cfg.epochs // 5) == 0 or ep == cfg.epochs - 1):
            print(f'  ep {ep:3d}  loss {tot/nb:.4f}  ({time.time()-t0:.0f}s)')
    return hist

@torch.no_grad()
def evaluate(model, data, cfg):
    """Rollout from t=0 for cfg.n_roll steps on held-out trajectories. Returns accuracy + structural diagnostics."""
    model.eval(); out = {}
    traj = data.to(DEV); n = min(cfg.n_roll, cfg.T)
    u0 = traj[:, 0]; true = traj[:, 1:1 + n]
    pred, zs, auxs = model.rollout(u0, n, create_graph=False, keep_aux=True)
    errs = [rel_l2(pred[:, t], true[:, t]).item() for t in range(n)]
    out['l2_1step'] = errs[0]; out['l2_rollout'] = float(np.mean(errs)); out['l2_final'] = errs[-1]
    # gauge-invariant dissipation of the FIXED physical energy Q (references no learned quantity):
    #   scalar PDEs: Q = 1/2||u||^2 ;  wave: Q = 1/2(||v||^2 + c^2||grad q||^2)   (Q of the observed field alone
    #   is not an invariant of the wave, cf. GENERIC-FNO Sec. 5.4).  r_mech = -<dQ/du, du> / (||dQ/du|| ||du||).
    prev = torch.cat([u0[:, None], pred[:, :-1]], 1).flatten(0, 1); du = (pred - prev.view_as(pred)).flatten(0, 1)
    # midpoint form: for a quadratic Q, -<dQ/du(u_mid), du> = Q(u) - Q(u') exactly, so an exactly Q-conserving step
    # (e.g. transport) gives r_mech = 0; the endpoint form has an O(|du|/|u|) bias (~0.1 at this frame spacing)
    _, gQ = phys_energy(0.5 * (prev + prev + du), cfg)
    out['rmech'] = (-ip(gQ, du) / (nrm(gQ) * nrm(du) + 1e-12)).mean().item()
    tprev = torch.cat([u0[:, None], true[:, :-1]], 1).flatten(0, 1)
    Qt0, _ = phys_energy(tprev, cfg); Qt1, _ = phys_energy(true.flatten(0, 1), cfg)
    out['Pi_star'] = ((Qt0 - Qt1) / (Qt0 + 1e-12)).mean().item()
    # same normalization as Pi_star (fractional one-step change of the fixed Q), so model and truth compare by MAGNITUDE,
    # not only by rank: r_mech is a cosine and under-weights dissipation on transport-dominated flows (Burgers)
    Qm0, _ = phys_energy(prev, cfg); Qm1, _ = phys_energy(pred.flatten(0, 1), cfg)
    out['Pi_model'] = ((Qm0 - Qm1) / (Qm0 + 1e-12)).mean().item()
    out['Q_ratio_final'] = (phys_energy(pred[:, -1], cfg)[0] / phys_energy(true[:, -1], cfg)[0]).mean().item()
    if pred.shape[2] == 1:                                   # scalar conservation laws: drift of int u along the rollout, relative to ||u0||
        m0 = u0.flatten(1).mean(1, keepdim=True); mt = pred[:, :, 0].flatten(2).mean(2)
        out['mass_drift'] = ((mt - m0).abs().amax(1) / (nrm(u0) + 1e-12)).mean().item()
        mtrue = true[:, :, 0].flatten(2).mean(2); out['mass_drift_true'] = ((mtrue - m0).abs().amax(1) / (nrm(u0) + 1e-12)).mean().item()
    if isinstance(model, GENERICFNO): out.update(model.diagnostics(u0, pred))
    if is_vlasov(cfg.pde): out.update(vlasov_metrics(pred, true, u0, cfg))
    if isinstance(model, CFO): out['nfe_per_frame'] = 4 * cfg.cfo_substeps
    if isinstance(model, PoissonGENERIC):
        if isinstance(model.E, PhysicalEnergy):    # Theorem: Q(u_t) <= E[z_0] - |Omega| inf phi   (= E[z_0] when phi >= 0)
            E0 = model.E(zs[:, 0], model.p)
            Qmax = torch.stack([phys_energy(pred[:, t], cfg)[0] for t in range(n)], 1).amax(1)
            phi_min = torch.stack([model.E.potential(pred[:, t]).flatten(1).amin(1) for t in range(n)], 1).amin(1).clamp(max=0)
            S0 = zs[:, 0, model.p:].flatten(1).mean(1)
            out['phi_min'] = phi_min.min().item()
            out['Qmax_over_E0'] = (Qmax / E0).max().item()
            out['Qmax_over_bound'] = (Qmax / (E0 - model.E.T0 * S0 - phi_min)).max().item()      # theorem: <= 1
        rE, dE, dS, sgn = [], [], [], []
        for t, aux in enumerate(auxs):
            dz = zs[:, t + 1] - zs[:, t]; g = aux.get('gEbar', aux['gE'])   # DG: test against the discrete gradient
            rE.append((ip(g, dz).abs() / (nrm(g) * nrm(dz) + 1e-30)).mean().item())
            E1, S1, _, _ = model.grads(zs[:, t + 1], create_graph=False)
            dE.append(((E1 - aux['E']).abs() / (aux['E'].abs() + 1e-12)).mean().item())
            dS.append((S1 - aux['S']).min().item())
        out['rE'] = float(np.mean(rE)); out['dE_rel'] = float(np.mean(dE)); out['min_dS'] = float(np.min(dS))
    return out

### 6b. Vlasov diagnostics (used by `evaluate` for the Vlasov PDEs)

In [ ]:
def vlasov_series(traj, cfg):
    """Invariants along (B, n, 1, nx, nx) -> dict of (B, n) tensors (physical units)."""
    B, n = traj.shape[:2]; g = vgrid(cfg.pde, traj.shape[-1], traj.device, traj.dtype)
    inv = g.invariants(traj.flatten(0, 1))
    return {k: v.view(B, n) for k, v in inv.items()}

def landau_rate(WE, dt, t_fit=8.0):
    """Damping rate from the envelope of W_E: least-squares slope of 1/2 log W_E through its local maxima in t <= t_fit
    (W_E oscillates at 2 omega about the decaying envelope |E| ~ exp(-gamma t)); per trajectory, then averaged."""
    n = min(WE.shape[1], int(round(t_fit / dt)) + 1); rates = []
    for b_ in range(WE.shape[0]):
        w = WE[b_, :n]; pk = [i for i in range(1, n - 1) if w[i] >= w[i - 1] and w[i] > w[i + 1]]
        if len(pk) < 2: pk = list(range(n))                      # no oscillation resolved: fall back to all frames
        t = torch.tensor(pk, device=WE.device, dtype=WE.dtype) * dt; y = 0.5 * w[pk].clamp_min(1e-300).log()
        t = t - t.mean(); rates.append((-((y - y.mean()) * t).sum() / (t ** 2).sum()).item())
    return float(np.mean(rates))

@torch.no_grad()
def vlasov_metrics(pred, true, u0, cfg):
    """Model-vs-truth on the Vlasov invariants over the rollout.  W_E (electric energy) is the Landau-damping / two-stream
    observable; H should be conserved (collisions included: Dougherty conserves energy); mass is a Casimir AND conserved by
    C[f]; S_B = -int f ln f is a Casimir produced by C[f] (min_dS >= 0 is the H-theorem); f >= 0 is not enforced by any model."""
    P = vlasov_series(torch.cat([u0[:, None], pred], 1), cfg); Tt = vlasov_series(torch.cat([u0[:, None], true], 1), cfg)
    out = {}
    out['WE_log10_err'] = (P['W_E'][:, 1:].clamp_min(1e-300).log10() - Tt['W_E'][:, 1:].clamp_min(1e-300).log10()).abs().mean().item()
    out['mass_drift'] = (P['mass'] / P['mass'][:, :1] - 1).abs().amax(1).mean().item()
    out['H_drift'] = (P['H'] / P['H'][:, :1] - 1).abs().amax(1).mean().item()
    out['H_drift_true'] = (Tt['H'] / Tt['H'][:, :1] - 1).abs().amax(1).mean().item()
    out['SB_min_dS'] = (P['S_B'][:, 1:] - P['S_B'][:, :-1]).amin(1).mean().item()
    out['SB_min_dS_true'] = (Tt['S_B'][:, 1:] - Tt['S_B'][:, :-1]).amin(1).mean().item()
    out['f_min'] = P['f_min'].amin().item()
    if cfg.pde == 'landau':
        out['landau_rate'] = landau_rate(P['W_E'], cfg.dt); out['landau_rate_true'] = landau_rate(Tt['W_E'], cfg.dt)
    return out

@torch.no_grad()
def plot_vlasov(models, cfg, n=None, snap_frac=(0.5, 1.0)):
    """models: {name: model}.  Electric energy, H, S_B, rollout error vs time, and f(x, v) snapshots (truth and models)."""
    data = get_data(cfg); _, te = split_data(data); te = te.to(DEV); n = n or min(cfg.n_roll, cfg.T)
    u0 = te[:, 0]; true = te[:, 1:1 + n]; t = np.arange(n + 1) * cfg.dt
    preds = {}
    for name, m in models.items():
        m.eval(); preds[name] = m.rollout(u0, n, create_graph=False)[0]
    ser = {'truth': vlasov_series(torch.cat([u0[:, None], true], 1), cfg)}
    ser.update({k: vlasov_series(torch.cat([u0[:, None], p], 1), cfg) for k, p in preds.items()})
    fig, ax = plt.subplots(1, 4, figsize=(17, 3.4))
    for name, s in ser.items():
        st = dict(c='k', ls='--') if name == 'truth' else {}
        ax[0].semilogy(t, s['W_E'].mean(0).cpu(), label=name, **st)
        ax[1].plot(t, (s['H'] / s['H'][:, :1]).mean(0).cpu(), label=name, **st)
        ax[2].plot(t, (s['S_B'] - s['S_B'][:, :1]).mean(0).cpu(), label=name, **st)
    if cfg.pde == 'landau':
        W0 = ser['truth']['W_E'][:, 0].mean().item()
        ax[0].semilogy(t, W0 * np.exp(-2 * LANDAU_THEORY['gamma'] * t), 'r:', label='linear theory, k=0.5')
    for name, p in preds.items():
        ax[3].semilogy(t[1:], [rel_l2(p[:, i], true[:, i]).item() for i in range(n)], label=name)
    ax[0].set(xlabel='t', ylabel='electric energy W_E'); ax[1].set(xlabel='t', ylabel='H(t)/H(0)')
    ax[2].set(xlabel='t', ylabel='S_B(t) - S_B(0)'); ax[3].set(xlabel='t', ylabel='relative L2')
    for a in ax: a.legend(fontsize=7)
    plt.tight_layout(); plt.show()
    cols = ['truth'] + list(preds); rows_ = [int(round(fr * n)) - 1 for fr in snap_frac]
    fig, ax = plt.subplots(len(rows_), len(cols), figsize=(3.2 * len(cols), 2.8 * len(rows_)), squeeze=False)
    g = vgrid(cfg.pde, cfg.nx, DEV, torch.float32); ext = [0, g.Lx, -PDE_PARAMS[cfg.pde]['vmax'], PDE_PARAMS[cfg.pde]['vmax']]
    for i, r in enumerate(rows_):
        for j, name in enumerate(cols):
            fld = (true if name == 'truth' else preds[name])[0, r, 0].T.cpu()
            ax[i, j].imshow(fld, origin='lower', aspect='auto', extent=ext, cmap='viridis', vmin=0, vmax=float(true[0, r, 0].max()))
            ax[i, j].set(title=f'{name}  t={t[r + 1]:.1f}', xlabel='x', ylabel='v')
    plt.tight_layout(); plt.show()


In [ ]:
def lie_poisson_jacobi_vs_resolution(band=8, sizes=(32, 48, 64, 128), dtype=torch.float64):
    """Jacobi residual of the discrete Lie-Poisson operator u D + D u on band-limited states with band-limited test
    functionals, as the grid is refined at fixed band.  Triple products need 3*band <= nx/2 to be alias-free."""
    torch.set_default_dtype(dtype); rows = []
    for nx in sizes:
        k = torch.fft.fftfreq(nx, d=1.0 / nx, device=DEV, dtype=dtype); mask = (k.abs() <= band)
        D = lambda x: torch.fft.ifft(1j * k * torch.fft.fft(x)).real
        P = lambda x: torch.fft.ifft(mask * torch.fft.fft(x)).real
        L = lambda zz, g: (zz[:, 0] * D(g[:, 0]) + D(zz[:, 0] * g[:, 0])).unsqueeze(1)
        z = P(torch.randn(2, nx, device=DEV, dtype=dtype)).unsqueeze(1)
        fs = []
        for _ in range(3):
            r = torch.stack([P(torch.randn(nx, device=DEV, dtype=dtype)) for _ in range(3)]); al = torch.randn(3, device=DEV, dtype=dtype)
            def fn(zz, r=r, al=al):
                N = zz[0].numel()
                return sum(al[j] * torch.tanh(ip(zz, r[j].view(1, 1, -1).expand_as(zz)) / math.sqrt(N)) for j in range(3)) + 0.5 * ip(zz, zz) / N
            fs.append(fn)
        rows.append(dict(nx=nx, band=band, resolved=(3 * band <= nx // 2), jacobi=jacobi_residual(z, fs, L)))
    torch.set_default_dtype(torch.float32)
    return pd.DataFrame(rows).set_index('nx')

_DATA_CACHE = {}
def get_data(cfg):
    key = (cfg.dim, cfg.pde, cfg.nx, cfg.n_traj, cfg.T, cfg.dt, cfg.kmax, cfg.seed)
    if key not in _DATA_CACHE:
        _DATA_CACHE[key] = gen_data(cfg).cpu()
    return _DATA_CACHE[key]

def run_one(cfg, verbose=True):
    seed_all(cfg.seed + 1)
    data = get_data(cfg); tr, te = split_data(data)
    model = make_model(cfg)
    if verbose: print(f'[{cfg.dim}D {cfg.pde:9s} {cfg.backbone:10s} {cfg.model:4s}] params={count_params(model):,}')
    t0 = time.time(); hist = train_model(model, tr, cfg, verbose)
    row = dict(dim=cfg.dim, pde=cfg.pde, backbone=cfg.backbone, model=cfg.model, seed=cfg.seed,
               params=count_params(model), train_s=time.time() - t0, final_loss=hist[-1])
    row.update(evaluate(model, te, cfg))
    # persistence reference
    te_d = te.to(DEV); n = min(cfg.n_roll, cfg.T)
    row['l2_persist'] = float(np.mean([rel_l2(te_d[:, 0], te_d[:, 1 + t]).item() for t in range(n)]))
    if verbose: print('   ', {k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items() if k not in ('dim','pde','backbone','model','seed')})
    return row, model

def band_limit(x, band):
    """Project (…, *grid) fields onto |k|_inf <= band."""
    dim = 1 if x.dim() <= 2 else x.dim() - 2 if x.dim() > 3 else 1
    d = tuple(range(-dim, 0)); nx = x.shape[-1]
    k = torch.fft.fftfreq(nx, d=1.0 / nx, device=x.device)
    ks = torch.meshgrid(*([k] * dim), indexing='ij'); mask = torch.stack([kk.abs() for kk in ks]).amax(0) <= band
    return torch.fft.ifftn(torch.fft.fftn(x, dim=d) * mask, dim=d).real

## B. Head-to-head on GENERIC-FNO's own testbed
GENERIC-FNO's data generators and frame spacing (dt = 0.01 / 0.005), its 200-sample, 20-frame protocol; every model sees the same data per seed. GENERIC-FNO trains with its own recipe (the published configuration), the baseline and our models with this paper's protocol. Reported under GENERIC-FNO's own metric (`l2_tmlr`, mean per-sample relative error, which its random amplitudes make noisy) and a batch-level relative L2 (`l2_global`), with persistence under both; plus the dissipation calibration. Writes `results_tmlr_testbed.csv` and prints LaTeX rows.


In [ ]:
# ## B. Head-to-head on GENERIC-FNO's own testbed
# Reviewer aasd asked for a direct comparison with GENERIC-FNO.  Table 3 of the paper makes it on this paper's testbed
# (dt = 0.1, unit-RMS band-8 fields), where GENERIC-FNO's single explicit Euler step is far from its comfort zone.  This
# section makes the same comparison on GENERIC-FNO's own testbed -- its data generators, its frame spacing (dt = 0.01 for
# heat and advection, 0.005 for Burgers), its 200-sample / 20-frame protocol -- where the published model is known to work.
# Every model sees the same data for a given seed; GENERIC-FNO is trained with its own recipe; our models and the FNO baseline
# with this paper's protocol.  Two accuracy metrics are reported, because GENERIC-FNO's own metric (mean over samples of the
# per-sample relative error) is dominated by occasional near-zero samples that its N(0, 0.5) amplitudes produce:
#   * `l2_tmlr`   -- GENERIC-FNO's metric: mean over 50 test samples and 10 steps of ||pred - true|| / ||true|| per sample;
#   * `l2_global` -- batch-level relative L2 per step, ||pred - true||_batch / ||true||_batch, averaged over 10 steps.
# Persistence is reported under both.  Pi_model / Pi_star and Q_10 / Q*_10 come from `evaluate` (fixed physical energy).
import math

QUICK_T    = True                                  # True: one seed (~20 min on a T4 with the four variants below); False: three seeds
SEEDS_T    = [0] if QUICK_T else [0, 1, 2]
PDES_T     = ['heat', 'advection', 'burgers']      # GENERIC-FNO's wave data observes u alone (its v is hidden), which no model here treats
VARIANTS_T = ['base', 'gfno-native', 'pgphysC', 'pgphys']   # add 'gfno' (GENERIC-FNO under our protocol) or 'pg' (learned-E,S) if wanted
TMLR_DT    = {'heat': 0.01, 'advection': 0.01, 'burgers': 0.005}
RES_T      = 'results_tmlr_testbed.csv'

VARIANTS_R = {                                      # as in the rebuttal notebook (Sec. R2-R3)
    'base':        dict(model='base'),
    'pg':          dict(model='pg'),
    'pgphys':      dict(model='pg', energy='physical', entropy='integral', m_type='state', integrator='split', train_integrator='split', n_sub=8, lie_poisson=True),
    'pgphysC':     dict(model='pg', energy='physical', entropy='integral', m_type='state', integrator='split', train_integrator='split', n_sub=8, lie_poisson=False),
    'gfno':        dict(model='gfno'),
    'gfno-native': dict(model='gfno-native'),
}

def _tmlr_ic(nx, n_lo, n_hi, kmax, amp, x):
    u0 = torch.zeros(nx)
    for _ in range(torch.randint(n_lo, n_hi, (1,)).item()):
        ki = torch.randint(1, kmax + 1, (1,)).item(); a = torch.randn(1).item() * amp; ph = torch.rand(1).item() * 2 * math.pi
        u0 += a * torch.sin(ki * x + ph)
    return u0

def tmlr_generate(pde, n_samples=200, nx=64, nt=20):
    """GENERIC-FNO's 1D generators (GENERIC_FNO_4_PDE_Experiment_in_1D.ipynb), same ICs, dt, nu, c, integrators. (n, nt+1, 1, nx)."""
    x = torch.linspace(0, 2 * math.pi, nx + 1)[:-1]; k = torch.fft.rfftfreq(nx, d=1.0 / nx); trajs = []
    for _ in range(n_samples):
        if pde == 'heat':
            dt, nu = 0.01, 0.01; u0 = _tmlr_ic(nx, 2, 6, nx // 4 - 1, 0.5, x); uh = torch.fft.rfft(u0); dec = torch.exp(-nu * k ** 2 * dt); tr = [u0.clone()]
            for _ in range(nt): uh = uh * dec; tr.append(torch.fft.irfft(uh, n=nx))
        elif pde == 'advection':
            dt, c = 0.01, 1.0; u0 = _tmlr_ic(nx, 2, 6, 6, 0.5, x); uh = torch.fft.rfft(u0); ph = torch.exp(-1j * c * k * dt); tr = [u0.clone()]
            for _ in range(nt): uh = uh * ph; tr.append(torch.fft.irfft(uh, n=nx))
        elif pde == 'burgers':
            dt, nu = 0.005, 0.02; u = _tmlr_ic(nx, 2, 5, 4, 0.3, x); tr = [u.clone()]
            for _ in range(nt):
                u = torch.fft.irfft(torch.fft.rfft(u) / (1 + nu * k ** 2 * dt), n=nx)
                u = u - dt * u * torch.fft.irfft(1j * k * torch.fft.rfft(u), n=nx); tr.append(u.clone())
        else: raise ValueError(pde)
        trajs.append(torch.stack(tr, 0))
    return torch.stack(trajs, 0).unsqueeze(2).float()

def rollout_metrics(model, test, n_rollout=10):
    """Both accuracy metrics for the model and for persistence, from frame 0 of the first min(50, n) test trajectories."""
    model.eval(); n_test = min(50, test.shape[0]); x0 = test[:n_test, 0].to(DEV)
    pred = model.rollout(x0, n_rollout, create_graph=False)[0].detach()          # (n_test, n_rollout, p, nx), every model here
    tm, gl, tm_p, gl_p = [], [], [], []
    for t in range(n_rollout):
        x = pred[:, t]; y = test[:n_test, t + 1].to(DEV)
        per = lambda a: (((a - y) ** 2).flatten(1).mean(1).sqrt() / ((y ** 2).flatten(1).mean(1).sqrt() + 1e-8)).mean().item()
        glo = lambda a: ((a - y).norm() / (y.norm() + 1e-12)).item()
        tm.append(per(x)); gl.append(glo(x)); tm_p.append(per(x0)); gl_p.append(glo(x0))
    return dict(l2_tmlr=float(np.mean(tm)), l2_global=float(np.mean(gl)), l2_tmlr_final=tm[-1], l2_global_final=gl[-1],
                persist_tmlr=float(np.mean(tm_p)), persist_global=float(np.mean(gl_p)))

def cfg_t(pde, variant, seed):
    c = Cfg(dim=1, pde=pde, nx=64, n_traj=200, T=20, dt=TMLR_DT[pde], kmax=8, m_L=16, m_M=16, width=24, modes=12, layers=3,
            epochs=60, batch=16, horizon=4, norm=False, backbone='fno', seed=seed)
    for k, v in VARIANTS_R.get(variant, {}).items(): setattr(c, k, v)
    if variant == 'pg': c.width = 12
    if variant.startswith('pgphys'): c.m_M = c.nx // 3
    if not variant.startswith('pgphys'): c.integrator = c.train_integrator = 'euler'
    return c

rows_t = pd.read_csv(RES_T).to_dict('records') if os.path.exists(RES_T) else []
done_t = {(r['pde'], r['model'], r['seed']) for r in rows_t}
MODELS_T = {}
for pde in PDES_T:
    for sd in SEEDS_T:
        seed_all(sd); data = tmlr_generate(pde); tr, te = split_data(data)          # one data set per (pde, seed), shared by all models
        for var in VARIANTS_T:
            if (pde, var, sd) in done_t: continue
            cfg = cfg_t(pde, var, sd); seed_all(sd + 1); model = make_model(cfg)
            print(f'[TMLR testbed  {pde:9s} {var:12s} seed {sd}] params={count_params(model):,}')
            t0 = time.time(); hist = train_model(model, tr, cfg, verbose=True)
            row = dict(pde=pde, model=var, seed=sd, params=count_params(model), train_s=round(time.time() - t0, 1), final_loss=hist[-1])
            row.update(evaluate(model, te, cfg)); row.update(rollout_metrics(model, te))
            print('    ', {k: round(v, 4) for k, v in row.items() if k in ('l2_tmlr', 'l2_global', 'persist_tmlr', 'persist_global', 'Pi_model', 'Pi_star', 'Q_ratio_final', 'train_s')})
            rows_t.append(row); MODELS_T[(pde, var, sd)] = model
            pd.DataFrame(rows_t).to_csv(RES_T, index=False)
df_t = pd.DataFrame(rows_t)
cols = ['l2_tmlr', 'l2_global', 'persist_tmlr', 'persist_global', 'Pi_model', 'Pi_star', 'Q_ratio_final', 'train_s']
summ_t = df_t.groupby(['pde', 'model'])[cols].agg(['mean', 'std']).round(4)
print('\nGENERIC-FNO testbed, mean (std) over seeds:'); display(summ_t)
print('\nPublished GENERIC-FNO (its metric, three seeds): heat 0.095  advection 0.233  Burgers 0.046')
# LaTeX rows for the paper (its metric and the global metric, mean +- std)
order = [v for v in ['base', 'gfno', 'gfno-native', 'pg', 'pgphysC', 'pgphys'] if v in VARIANTS_T]
for pde in PDES_T:
    cells_ = []
    for v in order:
        d = df_t[(df_t.pde == pde) & (df_t.model == v)]
        cells_.append(f"${d.l2_tmlr.mean():.3f}\\pm{d.l2_tmlr.std() if len(d) > 1 else 0:.3f}$ / ${d.l2_global.mean():.4f}\\pm{d.l2_global.std() if len(d) > 1 else 0:.4f}$")
    dp = df_t[(df_t.pde == pde) & (df_t.model == order[0])]
    print(f"{pde:9s} & ${dp.persist_tmlr.mean():.3f}$ / ${dp.persist_global.mean():.4f}$ & " + ' & '.join(cells_) + ' \\\\')


[TMLR testbed  heat      base         seed 0] params=44,593
  ep   0  loss 0.4194  (4s)
  ep  12  loss 0.0066  (7s)
  ep  24  loss 0.0047  (10s)
  ep  36  loss 0.0033  (12s)
  ep  48  loss 0.0027  (15s)
  ep  59  loss 0.0023  (18s)
     {'train_s': 24.6, 'Pi_star': 0.0159, 'Pi_model': 0.0158, 'Q_ratio_final': 1.0017, 'l2_tmlr': 0.0056, 'l2_global': 0.0086, 'persist_tmlr': 0.0528, 'persist_global': 0.058}
[TMLR testbed  heat      gfno-native  seed 0] params=88,036
  ep   0  mse 0.000041  (0s)
  ep  30  mse 0.000030  (6s)
  ep  60  mse 0.000021  (15s)
  ep  90  mse 0.000010  (23s)
  ep 120  mse 0.000011  (32s)
  ep 149  mse 0.000010  (39s)
     {'train_s': 39.1, 'Pi_star': 0.0159, 'Pi_model': -0.079, 'Q_ratio_final': 2.661, 'l2_tmlr': 0.2044, 'l2_global': 0.0682, 'persist_tmlr': 0.0528, 'persist_global': 0.058}
[TMLR testbed  heat      pgphysC      seed 0] params=18,702
  ep   0  loss 0.1535  (15s)
  ep  12  loss 0.0257  (110s)
  ep  24  loss 0.0172  (205s)
  ep  36  loss 0.0106  (299s)


l2_tmlr     l2_global     persist_tmlr      \
                         mean std      mean std         mean std   
pde       model                                                    
advection base         0.0009 NaN    0.0012 NaN       0.1886 NaN   
          gfno-native  0.3553 NaN    0.2069 NaN       0.1886 NaN   
          pgphys       0.0004 NaN    0.0011 NaN       0.1886 NaN   
          pgphysC      0.0003 NaN    0.0008 NaN       0.1886 NaN   
burgers   base         0.0011 NaN    0.0029 NaN       0.0183 NaN   
          gfno-native  0.0331 NaN    0.0317 NaN       0.0183 NaN   
          pgphys       0.0026 NaN    0.0211 NaN       0.0183 NaN   
          pgphysC      0.0020 NaN    0.0153 NaN       0.0183 NaN   
heat      base         0.0056 NaN    0.0086 NaN       0.0528 NaN   
          gfno-native  0.2044 NaN    0.0682 NaN       0.0528 NaN   
          pgphys       0.0442 NaN    0.0556 NaN       0.0528 NaN   
          pgphysC      0.0366 NaN    0.0175 NaN       0.0528 NaN   

                      persist_global     Pi_model     Pi_star      \
                                mean std     mean std    mean std   
pde       model                                                     
advection base                0.1978 NaN   0.0000 NaN  0.0000 NaN   
          gfno-native         0.1978 NaN  -0.0481 NaN  0.0000 NaN   
          pgphys              0.1978 NaN   0.0000 NaN  0.0000 NaN   
          pgphysC             0.1978 NaN   0.0000 NaN  0.0000 NaN   
burgers   base                0.0338 NaN   0.0016 NaN  0.0016 NaN   
          gfno-native         0.0338 NaN  -0.0009 NaN  0.0016 NaN   
          pgphys              0.0338 NaN   0.0014 NaN  0.0016 NaN   
          pgphysC             0.0338 NaN   0.0016 NaN  0.0016 NaN   
heat      base                0.0580 NaN   0.0158 NaN  0.0159 NaN   
          gfno-native         0.0580 NaN  -0.0790 NaN  0.0159 NaN   
          pgphys              0.0580 NaN   0.0310 NaN  0.0159 NaN   
          pgphysC             0.0580 NaN   0.0226 NaN  0.0159 NaN   

                      Q_ratio_final     train_s      
                               mean std    mean std  
pde       model                                      
advection base               0.9999 NaN    14.7 NaN  
          gfno-native        1.8244 NaN    39.3 NaN  
          pgphys             0.9999 NaN   518.2 NaN  
          pgphysC            1.0000 NaN   477.2 NaN  
burgers   base               0.9997 NaN    14.3 NaN  
          gfno-native        1.0256 NaN    38.9 NaN  
          pgphys             1.0015 NaN   521.0 NaN  
          pgphysC            1.0002 NaN   477.3 NaN  
heat      base               1.0017 NaN    24.6 NaN  
          gfno-native        2.6610 NaN    39.1 NaN  
          pgphys             0.9003 NaN   515.6 NaN  
          pgphysC            0.9574 NaN   480.6 NaN


Published GENERIC-FNO (its metric, three seeds): heat 0.095  advection 0.233  Burgers 0.046
heat      & $0.053$ / $0.0580$ & $0.006\pm0.000$ / $0.0086\pm0.0000$ & $0.204\pm0.000$ / $0.0682\pm0.0000$ & $0.037\pm0.000$ / $0.0175\pm0.0000$ & $0.044\pm0.000$ / $0.0556\pm0.0000$ \\
advection & $0.189$ / $0.1978$ & $0.001\pm0.000$ / $0.0012\pm0.0000$ & $0.355\pm0.000$ / $0.2069\pm0.0000$ & $0.000\pm0.000$ / $0.0008\pm0.0000$ & $0.000\pm0.000$ / $0.0011\pm0.0000$ \\
burgers   & $0.018$ / $0.0338$ & $0.001\pm0.000$ / $0.0029\pm0.0000$ & $0.033\pm0.000$ / $0.0317\pm0.0000$ & $0.002\pm0.000$ / $0.0153\pm0.0000$ & $0.003\pm0.000$ / $0.0211\pm0.0000$ \\
